# Chicago Crime Clustering

**Author:** Erik Pak  
**Date:** 04/2026  
**Data:** Chicago Data Portal - Crime Incidents 2001–2025

---
## Era Definitions

| Era | Period | Months |
|---|---|---|
| Pre-COVID | Jan 2001 – Feb 2020 | 230 |
| COVID | Mar 2020 – Dec 2022 | 34 |
| Post-COVID | Jan 2023 – Dec 2025 | 36 |

**Era cutoff rationale:** The COVID era begins in March 2020, coinciding with the Illinois stay-at-home order (March 21, 2020) and the WHO pandemic declaration (March 11, 2020). The post-COVID era begins in January 2023, following the expiration of Illinois's disaster proclamation and the effective end of major federal pandemic-era policies in late 2022. These boundaries are administrative and policy-based; the underlying behavioral and enforcement shifts may not align exactly with these dates.

---

## Methodology


In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
# import random
from importlib.metadata import version
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr, pearsonr

# Reproducibility
# SEED is applied to all random processes throughout the pipeline.
SEED = 1776

# Path + custom modules───
sys.path.append('../Src/')
import crime_era_clustering as hc
import analyze_fill_stats as af
import era_plot as plot

# Library versions
versions = {
    "Python"  : sys.version.split()[0],
    "Pandas"  : pd.__version__,
    "NumPy"   : np.__version__,
    "Pyarrow" : pa.__version__,
    "Seaborn" : sns.__version__,
    "Matplot" : sys.modules['matplotlib'].__version__,
    "Scipy"   : sys.modules['scipy'].__version__,
}
df_versions = pd.DataFrame(list(versions.items()), columns=['Library', 'Version'])
print(df_versions)

# Display settings
np.set_printoptions(suppress=True, precision=4, linewidth=100)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)  # No limit on cell content length

   Library Version
0   Python  3.13.9
1   Pandas   2.3.3
2    NumPy   2.3.4
3  Pyarrow  22.0.0
4  Seaborn  0.13.2
5  Matplot  3.10.7
6    Scipy  1.16.3


## Data Import

Chicago crime incident data was loaded from a PyArrow Feather file. The dataset contains **8,469,443 records** across 34 columns, spanning January 2001 through December 2025, after further processing by the `ChicagoCrimeEraAnalysis.ipynb` notebook.

In [2]:
# PyArrow's version of the 'arrow' backend
df = feather.read_feather('../Data/crime_data_covid.feather', memory_map=True, types_mapper=pd.ArrowDtype)
# display
df.head()

,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,beat,district,sector,ward,community_code,community_name,community_area,fbi_code,x_coordinate,y_coordinate,year,latitude,longitude,era,month,day_of_week,quarter,year_quarter,time_of_day,fbi_code_desc,fbi_index_code,district_location,year_week,year_month,Indexed
0,01G050460,2001-01-24 20:45:00,072XX S RIDGELAND AV,1811,NARCOTICS,POSS: CANNABIS 30GMS OR LESS,SIDEWALK,True,False,0324,003,1,<NA>,<NA>,<NA>,<NA>,18,1189075,1857566,2001,41.764219,-87.582549,pre_covid,January,Wednesday,Q1,2001-Q1,Night,Drug Abuse Violations,False,Grand Crossing,2001-04,200101,N
1,03J493690,2003-07-12 17:00:00,075XX S DOBSON AVE,0890,THEFT,FROM BUILDING,APARTMENT,False,False,0624,006,2,08,69,GREATER GRAND CROSSING,98853167.7093,06,1184198,1855214,2003,41.75788,-87.600498,pre_covid,July,Saturday,Q3,2003-Q3,Evening,Larceny – Theft,True,Gresham,2003-28,200307,I
2,04X245238,2004-12-13 21:15:00,006XX N RIDGEWAY AVE,2024,NARCOTICS,POSS: HEROIN(WHITE),SIDEWALK,True,False,1122,011,4,27,23,HUMBOLDT PARK,100480876.502,18,1151273,1903996,2004,41.892451,-87.719888,pre_covid,December,Monday,Q4,2004-Q4,Night,Drug Abuse Violations,False,Harrison,2004-51,200412,N
3,07C115980,2006-03-31 09:15:00,026XX N NARRAGANSETT AVE,0610,BURGLARY,FORCIBLE ENTRY,APARTMENT,False,False,2512,025,5,29,19,BELMONT CRAGIN,109099414.689,05,1133296,1916864,2006,41.928096,-87.78561,pre_covid,March,Friday,Q1,2006-Q1,Morning,Burglary,True,Grand Central,2006-13,200603,I
4,07HN36467,2007-05-25 14:51:00,022XX N LA CROSSE AVE,1812,NARCOTICS,POSS: CANNABIS MORE THAN 30GMS,RESIDENCE,True,False,2522,025,5,31,19,BELMONT CRAGIN,109099414.689,18,1143697,1914371,2007,41.921066,-87.747452,pre_covid,May,Friday,Q2,2007-Q2,Afternoon,Drug Abuse Violations,False,Grand Central,2007-21,200705,N


In [3]:
df.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8469443 entries, 0 to 8469442
Data columns (total 34 columns):
 #   Column                Non-Null Count    Dtype                                                       
---  ------                --------------    -----                                                       
 0   case_number           8469443 non-null  string[pyarrow]                                             
 1   date                  8469443 non-null  timestamp[s][pyarrow]                                       
 2   block                 8469443 non-null  string[pyarrow]                                             
 3   iucr                  8469443 non-null  string[pyarrow]                                             
 4   primary_type          8469443 non-null  string[pyarrow]                                             
 5   description           8469443 non-null  string[pyarrow]                                             
 6   location_description  8454105 non-

In [4]:
df.era.unique()

<ArrowExtensionArray>
['pre_covid', 'post_covid', 'covid']
Length: 3, dtype: dictionary<values=string, indices=int8, ordered=0>[pyarrow]

In [5]:
# Separate by era
pre_df   = df[df.era == 'pre_covid']
covid_df = df[df.era == 'covid']
post_df  = df[df.era == 'post_covid']
# display shape
pre_df.shape, covid_df.shape, post_df.shape

((7092647, 34), (623871, 34), (752925, 34))

## Data Preparation

Each raw record represents a single crime incident. To enable era-level analysis, records are aggregated to monthly crime counts per crime type using `groupby(['year_month', 'fbi_code_desc']).size()`. This produces a long-format dataframe with one row per crime type per month, which is the foundation for all downstream analysis.

The three era dataframes are then separated, missing crime/month combinations are identified and filled with zero (true zeros - months with no recorded incidents), and a datetime `date` column is parsed from the integer `year_month` field.

In [6]:
# Aggregate to monthly crime counts per crime type
pre_df   = pre_df.groupby(['year_month', 'fbi_code_desc']).size().reset_index(name="crime_count")
covid_df = covid_df.groupby(['year_month', 'fbi_code_desc']).size().reset_index(name="crime_count")
post_df  = post_df.groupby(['year_month', 'fbi_code_desc']).size().reset_index(name="crime_count")

In [7]:
# Quick check on each era
print("::: Pre-COVID :::")
print(pre_df.shape)
print(pre_df['year_month'].min(), pre_df['year_month'].max())

print("\n::: COVID :::")
print(covid_df.shape)
print(covid_df['year_month'].min(), covid_df['year_month'].max())

print("\n::: Post-COVID :::")
print(post_df.shape)
print(post_df['year_month'].min(), post_df['year_month'].max())

::: Pre-COVID :::
(5788, 3)
200101 202002

::: COVID :::
(846, 3)
202003 202212

::: Post-COVID :::
(891, 3)
202301 202512


In [8]:
import pandas as pd
from itertools import product


# Data Prep Helper Functions ────────────────────────────────────────────────────────────
def prepare_era_data(data_df, era_label):
    """
    Standardizes temporal columns and assigns era labels.

    Args:
        data_df (pd.DataFrame): Must contain 'year_month' as an integer (YYYYMM format).
        era_label (str): Label to assign to the 'era' column.

    Returns:
        pd.DataFrame: Copy of input with 'year_month' as a datetime and 'era' set.

    Raises:
        ValueError: If the year_month column is missing from the data.
    """
    # input validation added - missing column previously raised a bare KeyError
    if 'year_month' not in data_df.columns:
        raise ValueError("Missing 'year_month' column.")

    data = data_df.copy()
    data['year_month'] = pd.to_datetime(data_df['year_month'].astype(str), format='%Y%m')
    data['era'] = era_label
    return data


# 1. Find Missing Crime Months ────────────────────────────────────────────────
def find_missing_crime_months(data_df: pd.DataFrame, era_label: str):
    """
    Identifies gaps in a crime time-series dataset by comparing actual 
    observations against a complete grid of all possible month/crime pairs.

    This function uses a Cartesian product to define the 'ideal' state of the 
    data and a left-merge indicator to isolate specific missing combinations.

    Args:
        data_df (pd.DataFrame): Long-form dataframe containing crime counts. 
            Must include 'year_month' and 'fbi_code_desc'.
        era_label (str): Label for the specific time period (e.g., 'Pre-COVID') 
            used for grouping and logging.

    Returns:
        tuple: (expected_rows, actual_rows, missing_only)
            - expected_rows (int): Theoretical count (unique_months * unique_crimes).
            - actual_rows (int): The number of observations currently in the data.
            - missing_only (pd.DataFrame): Subset of the grid containing only the 
              missing month/crime combinations.

    Raises:
        ValueError: If 'year_month' or 'fbi_code_desc' are missing from data_df.
    """
    
    # 1. Defensive Validation: Ensure the data 'contract' is honored before processing
    required_cols = {'year_month', 'fbi_code_desc'}
    if not required_cols.issubset(data_df.columns):
        missing = required_cols - set(data_df.columns)
        raise ValueError(f"find_missing_crime_months: Missing required columns: {missing}")

    # 2. Data Preparation: Standardize dates and assign era metadata
    # Note: prepare_era_data is assumed to handle datetime conversion
    d = prepare_era_data(data_df, era_label)

    # 3. Define the Dimensions: Get unique categories to build the 'Ideal Grid'
    all_months = d['year_month'].unique()
    all_crimes = d['fbi_code_desc'].unique()

    # Calculate theoretical vs actual density
    expected_rows = len(all_months) * len(all_crimes)
    actual_rows = len(d)

    # 4. The Cartesian Product: Generate every possible combination of month and crime.
    # This creates the 'Rectangular' baseline that a clean time series requires.
    
    full_grid = pd.DataFrame(
        product(all_months, all_crimes),
        columns=['year_month', 'fbi_code_desc']
    )

    # 5. The Gap Analysis: Merge actual data onto the full grid.
    # We use 'indicator=True' to create the '_merge' column, which flags 
    # rows present in the grid but absent in the data ('left_only').
    missing_df = full_grid.merge(
        d[['year_month', 'fbi_code_desc']],
        on=['year_month', 'fbi_code_desc'],
        how='left',
        indicator=True
    )

    # 6. Filter results: Isolate only the 'holes' identified by the merge
    missing_only = missing_df.loc[
        missing_df['_merge'] == 'left_only', 
        ['year_month', 'fbi_code_desc']
    ].copy()

    return expected_rows, actual_rows, missing_only


# 2. Era Integrity Report ────────────────────────────────────────────────
def run_era_integrity_report(era_dict, expected_counts):
    """
    Orchestrates a multi-era data integrity audit to detect temporal gaps 
    and row-level density issues.

    Phase 1: Temporal Validation - Confirms every era contains the exact number 
             of expected months.
    Phase 2: Density Audit - Uses Cartesian product gaps to find missing 
             crime-month combinations and infer duplicates.

    Args:
        era_dict (dict): Maps era labels (str) to DataFrames.
        expected_counts (dict): Maps era labels (str) to expected unique month counts (int).

    Raises:
        KeyError: If an era label is missing from the expected_counts map.
        AssertionError: If an era's unique month count is incorrect.
    """
    
    # PHASE 1: TEMPORAL GATEKEEPER
    # Verify ALL eras first. If one is wrong, the entire study is compromised.
    era_months_actual = {}
    
    for label, raw_df in era_dict.items():
        if label not in expected_counts:
            raise KeyError(f"run_era_integrity_report: Missing expected_counts for '{label}'")

        actual_months   = raw_df['year_month'].nunique()
        expected_months = expected_counts[label]

        # Fail-fast: prevents downstream analysis on truncated or overlapping eras
        assert actual_months == expected_months, (
            f"Era month mismatch: '{label}' expected {expected_months}, got {actual_months}. "
            f"Verify date boundary filtering logic."
        )
        era_months_actual[label] = actual_months

    print(f"✅ Era month counts verified: {era_months_actual}")

    # PHASE 2: ROW-LEVEL DENSITY AUDIT
    # Header for the diagnostic dashboard
    print(f"\n{'Era':<15} | {'Exp. Rows':<12} | {'Act. Rows':<12} | {'Missing'}")
    print("-" * 58)
    
    for label, raw_df in era_dict.items():
        # Call the Cartesian product gap finder
        exp_rows, act_rows, missing_df = find_missing_crime_months(raw_df, label)
        
        # Infer duplicates: 
        # (Expected - Missing) = what the row count SHOULD be. 
        # Difference from actual = duplicates.
        actual_missing  = len(missing_df)
        duplicate_count = act_rows - (exp_rows - actual_missing)
        
        if duplicate_count > 0:
            print(f"⚠️ WARNING: {duplicate_count} duplicate rows detected in {label}")

        # Summary line
        print(f"{label:<15} | {exp_rows:<12} | {act_rows:<12} | {actual_missing}")

        # If gaps exist, provide a breakdown by crime category
        if not missing_df.empty:
            print(" └── 🔍 Gaps by crime category:")
            gaps = missing_df.groupby('fbi_code_desc').size().sort_values(ascending=False)
            print(gaps.to_string())
        print()


# 3. Fill Missing Crime Months ────────────────────────────────────────────────
def fill_missing(data_df: pd.DataFrame) -> pd.DataFrame:
    """
    Zero-filling for time-series gaps.
    
    This function "rectangularizes" the dataframe by ensuring every month within 
    the era's range contains a row for every crime category. Missing combinations 
    are treated as structural zeros (no crimes reported) rather than missing values.

    Args:
        data_df (pd.DataFrame): Input dataframe. Must contain 'year_month', 
            'fbi_code_desc', 'crime_count', and 'era'.
            'year_month' should be in a format that can be converted to PeriodIndex.

    Returns:
        pd.DataFrame: A zero-filled, sorted dataframe with a continuous 
            monthly timeline for every crime category.

    Raises:
        ValueError: If required columns are missing or if the dataframe 
            contains multiple eras.
    """

    # 1.Defensive Validation
    required_cols = {'year_month', 'fbi_code_desc', 'crime_count', 'era'}
    if not required_cols.issubset(data_df.columns):
        missing = required_cols - set(data_df.columns)
        raise ValueError(f"fill_missing: missing required columns: {missing}")

    if data_df.empty:
        return data_df

    # Verification: Logic expects a single study period (era) per call
    if data_df['era'].nunique() != 1:
        raise ValueError("fill_missing: expects exactly one era per call")

    #2. Data Normalization
    out = data_df.copy()

    # Convert to PeriodIndex for robust monthly interval arithmetic
    out['year_month'] = pd.PeriodIndex(out['year_month'], freq='M')
    
    # Cast to Category for significant memory and GroupBy performance gains
    out['fbi_code_desc'] = out['fbi_code_desc'].astype('category')

    # Capture era label before transformation
    era_label = out['era'].iat[0]

    # 3. Grid Construction
    # Generate a continuous range from min to max date (prevents entire missing months)
    months = pd.period_range(
        out['year_month'].min(),
        out['year_month'].max(),
        freq='M'
    )

    # Extract all categories (crimes) present in this era
    crimes = out['fbi_code_desc'].cat.categories

    # Create the theoretical Cartesian product (The "Ideal Grid")
    full_idx = pd.MultiIndex.from_product(
        [months, crimes],
        names=['year_month', 'fbi_code_desc']
    )

    # 4. Aggregation & Zero-Fill 
    # observed=False ensures that even categories with zero total counts appear.
    # sum() handles potential duplicate rows for the same month/crime category.
    filled = (
        out.groupby(['year_month', 'fbi_code_desc'], observed=False)['crime_count']
        .sum()
        .reindex(full_idx, fill_value=0)
        .reset_index()
    )

    # 5. Finalization
    filled['era'] = era_label
    
    # int32 is memory-efficient and more than sufficient for monthly crime counts
    filled['crime_count'] = filled['crime_count'].astype(np.int32)
    
    # Convert Period back to Timestamp for compatibility with plotting and stats libraries
    filled['year_month'] = filled['year_month'].dt.to_timestamp()

    # Stable sort ensures deterministic ordering for DTW and lagging operations
    return filled.sort_values(['year_month', 'fbi_code_desc'], kind='stable')

In [9]:
# Number of months per era: derive dynamically from feather file
era_months = (
    df.groupby('era')['year_month']
    .nunique()
    .to_dict()
)

# Build an era dict with DataFrame data
era_dict = {
    'pre_covid':  pre_df,
    'covid':      covid_df,
    'post_covid': post_df,
}

# Ground truth expectation
expected_era_months = {'pre_covid': 230, 'covid': 34, 'post_covid': 36}

# Era Completeness Analysis
run_era_integrity_report(era_dict, expected_era_months)

# New dictionary that bundles the data with its corresponding time index
era_data = {key: (data, era_months[key]) for key, data in era_dict.items()}

✅ Era month counts verified: {'pre_covid': 230, 'covid': 34, 'post_covid': 36}

Era             | Exp. Rows    | Act. Rows    | Missing
----------------------------------------------------------
pre_covid       | 5980         | 5788         | 192
 └── 🔍 Gaps by crime category:
fbi_code_desc
Involuntary Manslaughter / Reckless Homicide    187
Embezzlement                                      3
Gambling                                          2

covid           | 884          | 846          | 38
 └── 🔍 Gaps by crime category:
fbi_code_desc
Involuntary Manslaughter / Reckless Homicide    16
Gambling                                        13
Embezzlement                                     5
Prostitution                                     2
Stolen Property (Buy, Receive, Possess)          2

post_covid      | 936          | 891          | 45
 └── 🔍 Gaps by crime category:
fbi_code_desc
Involuntary Manslaughter / Reckless Homicide    27
Gambling                                        10
E

## Fill Missing with Zero

After identifying missing crime/month combinations in each era:

- **Involuntary Manslaughter** - missing 187/230 pre-COVID months. This crime is genuinely rare, so most months had zero incidents and simply were not recorded as a row.
- **Gambling, Embezzlement, Prostitution, Stolen Property** - sporadically missing for the same reason: rare months with zero counts.

These are **true zeros**, not missing data. The correct fix is to fill with 0, not drop these crimes - especially since Involuntary Manslaughter's rarity vs. spike pattern is exactly the kind of signal z-gap and r-spike are designed to catch.

In [10]:
# Rebuild era_data with prepared + filled dataframes
# ERA data -> preprocess -> fills missing month/crime combinations -> stores the cleaned version per era
era_data_filled = {}  # Create empty dictionary
for label, (raw_df, expected_months) in era_data.items(): # Loop through each era
    d = prepare_era_data(raw_df, label)                   # Clean the data: helper function
    d = fill_missing(d)                                   # Fill missing structure
    era_data_filled[label] = d                            # Store result
    print(f"{label}:  {len(d):,} rows after fill")

print("\n::: ✅ Ready for computing the baseline stats :::")

pre_covid:  5,980 rows after fill
covid:  884 rows after fill
post_covid:  936 rows after fill

::: ✅ Ready for computing the baseline stats :::


## Chicago Crime: Baseline (pre-COVID)

### Note

Involuntary Manslaughter has **MAD = 0** (Median Absolute Deviation = median(|each value − median of all values|)) because its median is 0, and most months are 0 as well. This means `robust_z` would involve division by zero for that crime. We already knew this crime needs special treatment - the binary presence rate will be its primary metric instead.

$$\text{Coefficient of Variation (CV)} = \frac{\sigma}{\mu} \times 100$$
$$\text{Robust CV (RSCV)} = \left( \frac{MAD}{Median} \right) \times 100$$

**High CV crimes produce unstable z-gaps** because elevated variability increases the baseline standard deviation, making real changes harder to distinguish from normal noise. This effect goes in both directions. High CV inflates the denominator, compressing z-gaps for truly significant shifts, while also allowing noisy fluctuations to occasionally produce large z-gaps that appear meaningful but are not.

**Low CV crimes yield more stable and interpretable z-gaps**, where values such as 2.0 are more likely to reflect genuine deviations from typical behavior rather than baseline noise.

**The Robust Coefficient of Variation (RSCV)** is a dispersion metric that uses "robust" statistics, which are less affected by extreme outliers or skewed data, instead of the traditional mean and standard deviation.
1. **Median:** The middle value of the dataset. Unlike the mean, it isn't pulled toward extreme spikes.
2. **MAD (Median Absolute Deviation):** The "robust" version of standard deviation. It measures the typical distance between each data point and the median.
    * Calculation: $MAD = \text{median}(|x_i - \text{median}(x)|)$
    * This converts the ratio into a percentage, making it easier to compare across different crime categories (e.g., comparing `Theft`, which has thousands of incidents, to `Homicide`, which has dozens).

### Explaination:

* `zero_mad` - True if MAD = 0 (division by zero risk) -> Involuntary Manslaughter only
* `cv` -  The Coefficient of Variation (CV) is the ratio of the standard deviation to the mean, a unitless measure of relative variability. It shows how large the variability is relative to the data's average level.
* `seasonal_strength` (Seasonal Strength ($F_s$)) - Seasonal strength measures how much of the variation follows a predictable, repeating pattern over a fixed period (e.g., 12 months).
    * Close to 1.0 -> strong, stable, repeating seasonal pattern
    * Around 0.4–0.6 -> moderate seasonality (pattern exists but is noisy)
    * Below 0.3 -> weak or no seasonality (crime type does not repeat in a predictable annual cycle)
        * Operationally:
            * High seasonal_strength -> STL decomposition is meaningful
            * Low seasonal_strength -> seasonality is not a reliable feature
            * Zero -> either no seasonality or STL failed / series too short 
* `trend_strength` (Trend Strength ($F_t$)) - Trend strength measures the extent to which the data follows a long-term direction (upward or downward) that isn't part of a repeating cycle.
    * Close to 1.0 -> strong, smooth, persistent trend
    * Around 0.4–0.6 -> moderate trend (trend exists but is noisy or interrupted)
    * Below 0.3 -> weak or no trend (series is stationary or dominated by noise)
        * Operationally:
            * High trend_strength -> decomposition is useful
            * Low trend_strength -> trend is not a meaningful structural feature
            * Zero -> either no trend or STL failed / series too short 

In [ ]:
import numpy as np
import pandas as pd
import warnings
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, zivot_andrews


# Global Constants for Volatility Analysis
_VOLATILITY_BINS        = [-np.inf, 25, 40, 60, np.inf]
_VOLATILITY_BINS_ROBUST = [-np.inf, 20, 35, 55, np.inf]  # shifted left — robust_cv runs ~8-10pts lower
_VOLATILITY_LABELS      = ["Very Stable", "Moderate", "Volatile", "Highly Volatile"]
_STL_MIN_MONTHS         = 24       # minimum months required for STL
_STL_SEASONAL_PERIOD    = 12       # monthly seasonality

# Global Configuration ADF
_ADF_ALPHA              = 0.05     # If the p-value < 0.05, reject the null—data is stationary (statistical hurdle)
_ADF_MAX_LAGS           = None
_ADF_REGRESSION         = "c"      # "c" means constant (baseline average); use “ct” if there’s a trend (constant + trend)
_ADF_AUTO_LAG           = "AIC"    # Akaike Information Criterion - AIC selects the optimal number of lags, balancing fit and complexity
_ADF_MIN_OBS            = 20       # DF relies on regression; with fewer than 20 points, results are unreliable
_ADF_CONST_THRESHOLD    = 1e-8     # Detects flatline series—near-zero variance indicates a constant, skip regression to avoid singular matrix errors
_ADF_MAX_LAG_MONTHLY    = 12       # Caps AIC-selected lags (e.g., at 12) to keep focus on seasonal patterns, not distant history
_LOG_TRANSFORM          = True

# is_adf_eligible
_ELIGIBILITY_MIN_MONTHS   = 24
_ELIGIBILITY_MIN_PRESENCE = 30
_ELIGIBILITY_MIN_MEAN     = 2
_ELIGIBILITY_MAX_CV       = 150

# ADF Eligibility Helper Function ──────────────────────────────────────────────────
def _is_adf_eligible(row):
    """
    Determines if a crime series has enough signal for reliable stationarity testing.
    """
    return (
        row["months"] >= _ELIGIBILITY_MIN_MONTHS and
        row["presence_rate"] >= _ELIGIBILITY_MIN_PRESENCE and
        row["mean"] >= _ELIGIBILITY_MIN_MEAN and
        row["cv"] < _ELIGIBILITY_MAX_CV
    )


#  Data Standardized Helper Function ──────────────────────────────────────────────────
def _clean_series(series):
    """
    Standardizes input types and removes NaNs for the ZA test.
    Returns (x, idx) where x is 1D float array, idx is the original index (or None).
    """
    if isinstance(series, pd.Series):
        series = series.sort_index()
        idx = series.index
        x = series.to_numpy()
    elif isinstance(series, pd.DataFrame):
        series = series.sort_index()
        idx = series.index
        x = series.iloc[:, 0].to_numpy()
    else:
        idx = None
        x = np.asarray(series)

    x = x.astype(float).flatten()
    x = x[~np.isnan(x)]
    return x, idx





def _stationarity(
    series,
    series_name="series",
    alpha=_ADF_ALPHA,
    max_lags=_ADF_MAX_LAGS,
    regression=_ADF_REGRESSION,
    autolag=_ADF_AUTO_LAG,
    min_obs=_ADF_MIN_OBS,
    const_threshold=_ADF_CONST_THRESHOLD,
    max_lag_monthly=_ADF_MAX_LAG_MONTHLY,
    log_transform=_LOG_TRANSFORM,
):
    """
    Robust Augmented Dickey-Fuller (ADF) test wrapper.

    Enhancements:
    - Constant-series handling (no misleading p-values)
    - Strong stationarity check via critical values
    - Trend detection heuristic
    - Lag ceiling detection
    - Low-power flag for small samples
    - Optional log1p transform for count data
    """

    # 0. Clean data
    clean = np.asarray(series).flatten()
    clean = clean[~np.isnan(clean)]

    n_raw   = len(series)
    n_clean = len(clean)

    # 1. Insufficient data
    if n_clean < min_obs:
        return {
            # "adf_series"            : series_name,
            "adf_status"            : "insufficient_data",
            "adf_stat"              : None,
            "adf_p_value"           : None,           # <-- stat then p_value
            "adf_n_lags"            : None,
            "adf_n_obs"             : n_clean,
            "adf_n_raw_obs"         : n_raw,
            "adf_crit_vals"         : None,
            "adf_icbest"            : None,
            "adf_is_stationary"     : None,
            "adf_strong_stationarity": None,
            "adf_trend_detected"    : None,
            "adf_hit_lag_ceiling"   : None,
            "adf_low_power"         : True,
            "adf_alpha"             : alpha,
            "adf_unstable"          : False,
            "adf_warnings"          : [f"ADF skipped: observations < {min_obs}"],
            "adf_error"             : None,
        }

    # 2. Constant / near-constant
    n_unique = len(np.unique(clean))
    if n_unique <= 1 or (n_unique > 1 and clean.std() < const_threshold):
        return {
            # "adf_series"            : series_name,
            "adf_status"            : "constant_series",
            "adf_stat"              : None,
            "adf_p_value"           : None,
            "adf_n_lags"            : 0,
            "adf_n_obs"             : n_clean,
            "adf_n_raw_obs"         : n_raw,
            "adf_crit_vals"         : None,
            "adf_icbest"            : None,
            "adf_is_stationary"     : True,            # trivially stationary
            "adf_strong_stationarity": True,
            "adf_trend_detected"    : False,
            "adf_hit_lag_ceiling"   : False,
            "adf_low_power"         : n_clean < 40,
            "adf_alpha"             : alpha,
            "adf_unstable"          : False,
            "adf_warnings"          : ["ADF skipped: constant or near-constant series."],
            "adf_error"             : None,
        }

    # 3. Optional log transform
    if log_transform and np.all(clean >= 0):
        clean = np.log1p(clean)

    # 4. Adaptive lag selection
    if max_lags is None:
        max_lags = int(min(max_lag_monthly, n_clean // 3))
    if max_lags < 1:
        max_lags = 1

    caught_warnings = []

    try:
        # 5. Run ADF
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            result = adfuller(
                clean,
                autolag=autolag,
                maxlag=max_lags,
                regression=regression,
            )

        caught_warnings = [str(wi.message) for wi in w]

        adf_stat, p_value, n_lags, n_obs, crit_vals, icbest = result

        # 6. Interpretation
        is_stationary = bool(p_value < alpha)

        crit_5 = crit_vals.get('5%')
        strong_stationarity = adf_stat < crit_5 if crit_5 is not None else None

        # 7. Diagnostics
        trend_slope = np.polyfit(np.arange(n_clean), clean, 1)[0]
        has_trend   = abs(trend_slope) > 1e-3

        hit_lag_ceiling = (n_lags == max_lags)
        low_power       = n_clean < 40
        adf_unstable    = bool(caught_warnings or hit_lag_ceiling)

        # 8. Return - stat then p_value,
        #    is_stationary right after p_value,
        #    strong_stationarity right after is_stationary
        return {
            # "adf_series"            : series_name,
            "adf_status"            : "ok",
            "adf_stat"              : float(adf_stat),
            "adf_p_value"           : float(p_value),
            "adf_is_stationary"     : is_stationary,         # next to p_value
            "adf_strong_stationarity": strong_stationarity,  # next to is_stationary
            "adf_n_lags"            : int(n_lags),
            "adf_n_obs"             : int(n_obs),
            "adf_n_raw_obs"         : n_raw,
            "adf_crit_vals"         : dict(crit_vals),
            "adf_icbest"            : float(icbest) if icbest is not None else None,
            "adf_trend_detected"    : bool(has_trend),
            "adf_hit_lag_ceiling"   : hit_lag_ceiling,
            "adf_low_power"         : low_power,
            "adf_alpha"             : alpha,
            "adf_unstable"          : adf_unstable,
            "adf_warnings"          : caught_warnings,
            "adf_error"             : None,
        }

    except Exception as e:
        return {
            # "adf_series"            : series_name,
            "adf_status"            : "error",
            "adf_stat"              : None,
            "adf_p_value"           : None,
            "adf_is_stationary"     : None,
            "adf_strong_stationarity": None,
            "adf_n_lags"            : None,
            "adf_n_obs"             : n_clean,
            "adf_n_raw_obs"         : n_raw,
            "adf_crit_vals"         : None,
            "adf_icbest"            : None,
            "adf_trend_detected"    : None,
            "adf_hit_lag_ceiling"   : None,
            "adf_low_power"         : n_clean < 40,
            "adf_alpha"             : alpha,
            "adf_unstable"          : True,
            "adf_warnings"          : caught_warnings,
            "adf_error"             : str(e),
        }


def compute_baseline_stats(data_df: pd.DataFrame) -> pd.DataFrame:
    """
    Computes diagnostic metrics including Standard CV, Robust CV,
    sparsity flags, and STL decomposition strengths using global thresholds.
    """

    # 1. Validation: Ensures the dataframe isn't missing the core temporal or target keys
    required_cols = {'year_month', 'fbi_code_desc', 'crime_count'}
    if not required_cols.issubset(data_df.columns):
        raise ValueError(f"Missing columns: {required_cols - set(data_df.columns)}")

    # 2. Central Tendency & Dispersion: agg() is more efficient than calling mean/std separately
    baseline = (
        data_df.groupby('fbi_code_desc')['crime_count']
        .agg(
            mean='mean',
            std='std',
            median='median',
            months='count',
            mad=lambda x: (x - x.median()).abs().median(), # Median Absolute Deviation
        )
        .reset_index()
    )

    # 3. Volatility Metrics: np.where avoids ZeroDivisionErrors by checking if mean/median > 0
    baseline['cv'] = np.where(
        baseline['mean'] > 0,
        (baseline['std'] / baseline['mean']) * 100,
        np.nan
    ).round(1)

    baseline['robust_cv'] = np.where(
        baseline['median'] > 0,
        (baseline['mad'] / baseline['median']) * 100,
        np.nan
    ).round(1)

    # 4. Presence Rate: Determines "sparsity" (percentage of time period where crime actually occurred)
    total_months = data_df['year_month'].nunique()
    presence = (
        data_df[data_df['crime_count'] > 0]
        .groupby('fbi_code_desc')['year_month']
        .nunique()
        .reset_index(name='months_present')
    )
    presence['presence_rate'] = (
        presence['months_present'] / total_months * 100
    ).round(1)

    baseline = baseline.merge(presence, on='fbi_code_desc', how='left')
    # fillna(0) for crimes that never occurred in the baseline period
    baseline[['months_present', 'presence_rate']] = (
        baseline[['months_present', 'presence_rate']].fillna(0)
    )

    # Sparse mask: Categories with fewer than 24 months of activity are ineligible for STL
    _SPARSE_MASK = baseline['months_present'] < _STL_MIN_MONTHS

    # 5. Flagging Logic: Converts numerical metrics into readable diagnostic labels
    baseline["cv_flag"] = pd.cut(
        baseline["cv"],
        bins=_VOLATILITY_BINS,
        labels=_VOLATILITY_LABELS,
        right=False
    )
    # Categorical handling: We must explicitly add 'undefined' and 'Sparse' to the allowed set
    baseline["cv_flag"] = (
        baseline["cv_flag"]
        .cat.add_categories(["undefined", "Sparse"])
        .fillna("undefined")  # Flags math failures where the mean was zero
    )
    # Sparsity override: If the series is sparse, the volatility bin is scientifically irrelevant
    baseline.loc[_SPARSE_MASK, "cv_flag"] = "Sparse"

    # Robust CV Flagging: Same logic as above, but using shifted "Robust" thresholds
    baseline["robust_cv_flag"] = pd.cut(
        baseline["robust_cv"],
        bins=_VOLATILITY_BINS_ROBUST,
        labels=_VOLATILITY_LABELS,
        right=False
    )
    baseline["robust_cv_flag"] = (
        baseline["robust_cv_flag"]
        .cat.add_categories(["undefined", "Sparse"])
        .fillna("undefined")
    )
    baseline.loc[_SPARSE_MASK, "robust_cv_flag"] = "Sparse"
    

    # 6. STL Decomposition Strengths: Quantifies Trend and Seasonality vs Residual Noise
    def _stl_strengths(sub):
        ts = sub.sort_values('year_month')['crime_count'].to_numpy(dtype=float)
        # STL requires variance and a minimum data length (usually 2 seasonal periods)
        if len(ts) < _STL_MIN_MONTHS or np.all(ts == ts[0]):
            return pd.Series({
                'seasonal_strength': 0.0,
                'trend_strength':    0.0,
                'stl_warning':       "Insufficient Variance/Length."
            })
        try:
            with warnings.catch_warnings(record=True) as w:
                warnings.simplefilter("always")
                fit = STL(ts, period=_STL_SEASONAL_PERIOD, robust=True).fit()
                var_R = np.nanvar(fit.resid)

                # Strength formula: 1 - Var(Remainder) / Var(Remainder + Component)
                def _calc(comp):
                    var_RC = np.nanvar(fit.resid + comp)
                    return float(np.clip(1 - (var_R / var_RC), 0.0, 1.0)) if var_RC > 0 else 0.0

                return pd.Series({
                    'seasonal_strength': _calc(fit.seasonal),
                    'trend_strength':    _calc(fit.trend),
                    'stl_warning':       str(w[-1].message) if w else None
                })
        except Exception as e:
            return pd.Series({
                'seasonal_strength': 0.0,
                'trend_strength':    0.0,
                'stl_warning':       str(e)
            })

    # Apply the STL logic group-by-group across the fbi_code categories
    strengths = (
        data_df.groupby('fbi_code_desc')
        .apply(_stl_strengths, include_groups=False)
        .reset_index()
    )

    # New Baseline DataFrame
    baseline = baseline.merge(strengths, on='fbi_code_desc', how='left')

    
    # 8. Run the test for every crime type in your baseline dataframe
    baseline['adf_eligible'] = baseline.apply(_is_adf_eligible, axis=1)

    # 8. Run ADF only on Eligible series
    adf_list = []
    eligible_baseline = baseline[baseline['adf_eligible']].copy()
    
    for crime_type in eligible_baseline['fbi_code_desc']:
        series = data_df[data_df['fbi_code_desc'] == crime_type].sort_values('year_month')['crime_count']
        res = _stationarity(series, series_name=crime_type)
        res['fbi_code_desc'] = crime_type
        adf_list.append(res)

    if adf_list:
        adf_results = pd.DataFrame(adf_list)
        baseline = baseline.merge(adf_results, on='fbi_code_desc', how='left')

    return baseline

In [82]:
import warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import zivot_andrews

# -------------------------------------------------
# Clean + standardize input
# -------------------------------------------------
def _clean_series_for_za(series):
    """
    Standardizes input types and removes NaNs for the ZA test.
    Note: idx is None for raw arrays — break_date will be None
          even if break_index is valid.
    """
    if isinstance(series, pd.Series):
        series = series.sort_index()
        idx = series.index
        x = series.to_numpy()
    elif isinstance(series, pd.DataFrame):
        series = series.sort_index()
        idx = series.index
        x = series.iloc[:, 0].to_numpy()
    else:
        idx = None
        x = np.asarray(series)

    x = x.astype(float).flatten()
    x = x[~np.isnan(x)]

    return x, idx


# -------------------------------------------------
# Zivot-Andrews test (Full Production Version)
# -------------------------------------------------
_VALID_REGRESSION = {"c", "t", "ct"}

def za_test(
    series,
    name="series",
    alpha=0.05,
    min_obs=30,
    regression="c",   # "c" = constant break, "t" = trend, "ct" = both
    log1p_if_nonnegative=True,
):
    """
    Robust Zivot-Andrews test wrapper designed for long-term (230+ month)
    crime data with structural breaks.

    Fixes vs previous version:
        FIX 1 — Removed pre-demeaning: ZA handles constant internally via
                 regression param. Double-removing biased stat toward stationarity.
        FIX 2 — p-value guarded for NaN: ZA asymptotic p-value is unreliable;
                 stationarity now determined by critical values (stat < c5).
        FIX 3 — regression parameter validated before test runs.
    """

    # -----------------------------
    # Input validation
    # -----------------------------
    if regression not in _VALID_REGRESSION:
        raise ValueError(
            f"regression must be one of {_VALID_REGRESSION}, got '{regression}'"
        )

    x, idx = _clean_series_for_za(series)
    n = len(x)

    # -----------------------------
    # Output template
    # -----------------------------
    def base(status):
        return {
            "series"            : name,
            "status"            : status,
            "za_stat"           : None,
            "za_p_value"        : None,
            "za_stationary"     : None,
            "za_crit_vals"      : {"1%": None, "5%": None, "10%": None},
            "za_pass_1pct"      : False,
            "za_pass_5pct"      : False,
            "za_pass_10pct"     : False,
            "za_break_index"    : None,
            "za_break_date"     : None,
            "za_break_direction": None,
            "za_n_obs"          : n,
            "za_low_power"      : n < 40,
            "za_warnings"       : [],
            "error_type"        : None,
            "error"             : None,
        }

    # -----------------------------
    # 1. Guards: Empty, Insufficient, Constant
    # -----------------------------
    if n == 0:
        return base("all_missing")

    if n < min_obs:
        b = base("insufficient_data")
        b["za_warnings"].append(f"n={n} < min_obs={min_obs}")
        return b

    if np.std(x) < 1e-8:
        b = base("constant_series")
        b["za_stationary"] = True
        return b

    # -----------------------------
    # 2. Data Quality: Presence Rate & Spike Detection
    # -----------------------------
    presence_rate = np.mean(x > 0)
    if presence_rate < 0.20:
        b = base("zero_inflated")
        b["za_warnings"].append(f"Low presence_rate={presence_rate:.3f}")
        return b

    if np.any(x > 0):
        q75, q25 = np.percentile(x[x > 0], [75, 25])
        iqr = q75 - q25
        if iqr > 0:
            spike_score = np.max(x) / (iqr + 1e-8)
            if spike_score > 12:
                b = base("spike_driven")
                b["za_warnings"].append("Extreme outlier dominance detected.")
                return b

    # -----------------------------
    # 3. Transformation
    #    FIX 1: no pre-demeaning — ZA handles constant internally
    # -----------------------------
    if log1p_if_nonnegative and np.all(x >= 0):
        x = np.log1p(x)

    # -----------------------------
    # 4. Run Zivot-Andrews
    # -----------------------------
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            stat, p, crit, bp = zivot_andrews(x, regression=regression)

        # -----------------------------
        # 5. Critical values
        # -----------------------------
        c1  = crit.get("1%")
        c5  = crit.get("5%")
        c10 = crit.get("10%")

        # -----------------------------
        # 6. FIX 2: p-value guarded for NaN
        #    Stationarity determined by critical values, not p-value
        #    ZA asymptotic p-value is known to be unreliable
        # -----------------------------
        p_safe        = float(p) if (p is not None and not np.isnan(p)) else None
        is_stationary = bool(stat < c5) if c5 is not None else None

        # -----------------------------
        # 7. Break metadata
        # -----------------------------
        break_date = None
        if idx is not None and bp is not None and bp < len(idx):
            break_date = idx[bp]
            if hasattr(break_date, 'strftime'):
                break_date = break_date.strftime('%Y-%m')

        direction = None
        if bp is not None and 0 < bp < n - 1:
            direction = "up" if np.mean(x[bp:]) > np.mean(x[:bp]) else "down"

        # -----------------------------
        # 8. Return
        # -----------------------------
        return {
            "series"            : name,
            "status"            : "ok",
            "za_stat"           : float(stat),
            "za_p_value"        : p_safe,           # None if NaN
            "za_stationary"     : is_stationary,    # based on c5, not p-value
            "za_crit_vals"      : {
                "1%"  : float(c1)  if c1  is not None else None,
                "5%"  : float(c5)  if c5  is not None else None,
                "10%" : float(c10) if c10 is not None else None,
            },
            "za_pass_1pct"      : bool(stat < c1)  if c1  is not None else False,
            "za_pass_5pct"      : bool(stat < c5)  if c5  is not None else False,
            "za_pass_10pct"     : bool(stat < c10) if c10 is not None else False,
            "za_break_index"    : int(bp) if bp is not None else None,
            "za_break_date"     : break_date,
            "za_break_direction": direction,
            "za_n_obs"          : n,
            "za_low_power"      : n < 40,
            "za_warnings"       : [],
            "error_type"        : None,
            "error"             : None,
        }

    except Exception as e:
        b = base("error")
        b["error_type"] = e.__class__.__name__
        b["error"]      = str(e)
        return b

In [81]:
baseline_df = compute_baseline_stats(era_data_filled['pre_covid'])
baseline_df

,fbi_code_desc,mean,std,median,months,mad,cv,robust_cv,months_present,presence_rate,cv_flag,robust_cv_flag,seasonal_strength,trend_strength,stl_warning,adf_eligible,adf_status,adf_stat,adf_p_value,adf_is_stationary,adf_strong_stationarity,adf_n_lags,adf_n_obs,adf_n_raw_obs,adf_crit_vals,adf_icbest,adf_trend_detected,adf_hit_lag_ceiling,adf_low_power,adf_alpha,adf_unstable,adf_warnings,adf_error
0,Aggravated Assault,499.130435,118.601091,491.5,230,84.5,23.8,17.2,230,100.0,Very Stable,Very Stable,0.894673,0.926083,NaN,True,ok,-1.716439,0.422663,False,False,12.0,217.0,230.0,"{'1%': -3.460849270544952, '5%': -2.87495318813585, '10%': -2.5739190539191745}",-410.101375,True,True,False,0.05,True,[],None
1,Aggravated Battery,820.282609,288.874859,755.0,230,186.0,35.2,24.6,230,100.0,Moderate,Moderate,0.947459,0.966802,NaN,True,ok,-2.916648,0.043446,True,True,12.0,217.0,230.0,"{'1%': -3.460849270544952, '5%': -2.87495318813585, '10%': -2.5739190539191745}",-403.262625,True,True,False,0.05,True,[],None
2,Arson,50.595652,20.311052,46.0,230,14.0,40.1,30.4,230,100.0,Volatile,Moderate,0.495092,0.847806,NaN,True,ok,-1.510019,0.528549,False,False,12.0,217.0,230.0,"{'1%': -3.460849270544952, '5%': -2.87495318813585, '10%': -2.5739190539191745}",-68.188059,True,True,False,0.05,True,[],None
3,Burglary,1741.721739,562.889660,1869.0,230,466.5,32.3,25.0,230,100.0,Moderate,Moderate,0.913986,0.979162,NaN,True,ok,1.640215,0.997973,False,False,12.0,217.0,230.0,"{'1%': -3.460849270544952, '5%': -2.87495318813585, '10%': -2.5739190539191745}",-460.263250,True,True,False,0.05,True,[],None
4,Criminal Sexual Assault,155.343478,30.346461,151.0,230,20.5,19.5,13.6,230,100.0,Very Stable,Very Stable,0.801520,0.610042,NaN,True,ok,-1.736840,0.412238,False,False,11.0,218.0,230.0,"{'1%': -3.460707667106296, '5%': -2.874891213486339, '10%': -2.573885987711472}",-279.969139,False,False,False,0.05,False,[],None
5,Disorderly Conduct,285.647826,73.270822,272.5,230,55.5,25.7,20.4,230,100.0,Moderate,Moderate,0.789998,0.831140,NaN,True,ok,-1.106059,0.712649,False,False,12.0,217.0,230.0,"{'1%': -3.460849270544952, '5%': -2.87495318813585, '10%': -2.5739190539191745}",-264.078086,False,True,False,0.05,True,[],None
6,Drug Abuse Violations,3174.447826,1353.784491,3525.0,230,1042.0,42.6,29.6,230,100.0,Volatile,Moderate,0.404864,0.975107,NaN,True,ok,-0.274314,0.929064,False,False,12.0,217.0,230.0,"{'1%': -3.460849270544952, '5%': -2.87495318813585, '10%': -2.5739190539191745}",-420.420976,True,True,False,0.05,True,[],None
7,Embezzlement,6.543478,3.911167,6.0,230,2.0,59.8,33.3,227,98.7,Volatile,Moderate,0.511750,0.537763,NaN,True,ok,-1.517085,0.525043,False,False,8.0,221.0,230.0,"{'1%': -3.4602906385073884, '5%': -2.874708679520702, '10%': -2.573788599127782}",288.677260,True,False,False,0.05,False,[],None
8,Forgery and Counterfeiting,170.095652,42.921574,166.0,230,32.0,25.2,19.3,230,100.0,Moderate,Very Stable,0.394792,0.811386,NaN,True,ok,-1.025540,0.743836,False,False,11.0,218.0,230.0,"{'1%': -3.460707667106296, '5%': -2.874891213486339, '10%': -2.573885987711472}",-226.156380,True,False,False,0.05,False,[],None
9,Fraud,1084.108696,241.008147,988.0,230,107.0,22.2,10.8,230,100.0,Very Stable,Very Stable,0.397646,0.935778,NaN,True,ok,0.036044,0.961413,False,False,12.0,217.0,230.0,"{'1%': -3.460849270544952, '5%': -2.87495318813585, '10%': -2.5739190539191745}",-538.950439,True,True,False,0.05,True,[],None


In [ ]:
za_eligible = (
    months >= 36 and
    presence_rate >= 40 and
    mean >= 3
)

In [ ]:
def classify_series(adf_res, za_res):
    a = adf_res.get("adf_stationary")
    z = za_res.get("za_stationary")

    if a and z:
        return "stationary"
    if (not a) and z:
        return "regime_break"
    if (not a) and (not z):
        return "evolving"
    return "unstable"

In [21]:
%who

SEED	 STL	 StandardScaler	 adfuller	 af	 compute_baseline_stats	 covid_df	 d	 df	 
df_versions	 era_data	 era_data_filled	 era_dict	 era_months	 expected_era_months	 expected_months	 feather	 fill_missing	 
find_missing_crime_months	 hc	 is_adf_eligible	 k	 label	 mask	 np	 pa	 pd	 
pearsonr	 plot	 plt	 post_df	 pre_df	 prepare_era_data	 product	 raw_df	 run_era_integrity_report	 
series	 sns	 spearmanr	 sys	 test_stationarity	 v	 version	 versions	 warnings	 



| ADF Baseline   | ADF ZA (Zivot–Andrews) | Interpretation                           | Process Type                      |
| -------------- | ---------------------- | ---------------------------------------- | --------------------------------- |
| Stationary     | Stationary             | Stable baseline                          | Stable stationary process         |
| Non‑stationary | Stationary             | One structural break                     | Regime‑switching / break process  |
| Non‑stationary | Non‑stationary         | Drift / evolution                        | Evolving (non‑stationary) process |
| Unreliable     | Unreliable             | Sparse noise / non‑time‑series structure | Non‑time‑series or noisy data     |

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


def full_stationarity_report(series, series_name="series"):
    """
    Stationarity diagnostic: rolling stats + ADF test + ACF/PACF.
    Returns: dict with adf_stat, p_value, is_stationary, critical_values
    """
    # --- pre-processing ---
    # Principal's Note: Ensure we have enough data to actually run a 12-month rolling window
    data = series.dropna()
    if len(data) < 13:
        print(f"Error: Series '{series_name}' is too short for a 12-month rolling window.")
        return None

    # --- ADF test first so result can annotate the plot ---
    # autolag='AIC' is used to balance model complexity and goodness of fit
    adf_result = adfuller(data, autolag='AIC')
    adf_stat = adf_result[0]
    p_value = adf_result[1]
    n_lags = adf_result[2]
    n_obs = adf_result[3]
    crit_vals = adf_result[4]
    
    is_stat = p_value <= 0.05
    status_txt = "STATIONARY" if is_stat else "NON-STATIONARY"
    status_col = "#2ECC71" if is_stat else "#E74C3C" 

    # --- print console report ---
    dfoutput = pd.Series(
        [adf_stat, p_value, n_lags, n_obs],
        index=['ADF Statistic', 'p-value', 'Lags Used', 'Obs Used']
    )
    for k, v in crit_vals.items():
        dfoutput[f"Critical Value ({k})"] = v
        
    print(f"\n{'='*45}")
    print(f" ADF Report - {series_name}")
    print(f"{'='*45}")
    print(dfoutput.to_string())
    print(f"\n VERDICT: {series_name} is {status_txt} (p={p_value:.4f})")
    print(f"{'='*45}\n")

    # --- figure: 2x2 grid ---
    fig = plt.figure(figsize=(16, 10))
    gs = plt.GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.2)

    # Top: raw series + rolling stats
    ax1 = fig.add_subplot(gs[0, :])
    rolmean = data.rolling(window=12).mean()
    rolstd = data.rolling(window=12).std()
    
    ax1.plot(data, color='#3498DB', label='Original', alpha=0.5)
    ax1.plot(rolmean, color='#E74C3C', label='Rolling Mean (12)', linewidth=2.5)
    ax1.plot(rolstd, color='#2ECC71', label='Rolling Std (12)', linewidth=2)
    
    # Verdict Annotation
    ax1.annotate(
        f"ADF p={p_value:.4f} -> {status_txt}",
        xy=(0.01, 0.90), xycoords='axes fraction',
        fontsize=12, fontweight='bold', color=status_col,
        bbox=dict(boxstyle='round,pad=0.5', fc='white', alpha=0.9, ec=status_col)
    )
    ax1.set_title(f"Time Series Analysis: {series_name}", fontsize=14, pad=10)
    ax1.legend(loc='upper right', frameon=True)

    # Bottom Left: ACF
    ax2 = fig.add_subplot(gs[1, 0])
    plot_acf(data, lags=40, ax=ax2, zero=False, alpha=0.05)
    ax2.set_title("ACF - Total Autocorrelation (Detects Trend/Seasonality)", fontsize=12)

    # Bottom Right: PACF
    ax3 = fig.add_subplot(gs[1, 1])
    plot_pacf(data, lags=40, ax=ax3, zero=False, alpha=0.05, method='ywm')
    ax3.set_title("PACF - Direct Autocorrelation (Isolates Direct Lag Effect)", fontsize=12)

    fig.suptitle(f"Comprehensive Stationarity Diagnostic - {series_name}", fontsize=16, fontweight='bold', y=0.98)
    
    # Tight layout and display
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

    # --- Return dict ---
    return {
        'crime_type': series_name,
        'adf_stat': adf_stat,
        'p_value': p_value,
        'is_stationary': is_stat,
        'critical_values': crit_vals
    }

In [ ]:
# loop over all crime types and collect results
results = {}
for crime in pre_df.columns:
    results[crime] = full_stationarity_report(df[crime], series_name=crime)
    
stationary_crimes = [k for k, v in results.items() if v['is_stationary']]

### The baseline is a two-tier story
> The pre-COVID baseline separates into two dominant modeling regimes. Eighteen crimes exhibit strong trend or seasonal structure and are routed to STL decomposition, ensuring deviations are measured against cyclical-adjusted baselines rather than raw means. Six crimes with stable, stationary baselines are routed to the standard z-gap. The remaining two, one sparse, one high-volatility, are handled by presence tracking and robust z-score, respectively. This structure ensures seasonal and trend artifacts do not masquerade as substantive regime shifts in downstream analysis.

In [ ]:
import warnings
from statsmodels.tsa.stattools import kpss
from statsmodels.tools.sm_exceptions import InterpolationWarning

# 1. Isolate the pre-COVID data from your era_data_filled dictionary
pre_df = era_data_filled['pre_covid'].copy()

# 2. Pivot the data: Rows = Time, Columns = Crime Types
# This transforms your long data into a wide time-series format
df_pivot = pre_df.pivot(index='year_month', columns='fbi_code_desc', values='crime_count')

# 3. Define the 'Standard' candidates (Those you aren't decomposing yet)
standard_candidates = baseline_df.loc[baseline_df.use_decomp == False, 'fbi_code_desc'].to_list()

def stationarity_audit(df_wide, crimes):
    results = []
    for crime in crimes:
        if crime in df_wide.columns:
            series = df_wide[crime].dropna()
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                stat, p_value, lags, crit = kpss(series, regression='c')
            warning_msg = str(caught[0].message) if caught else None
            results.append({
                'Crime':     crime,
                'KPSS_Stat': round(stat, 4),
                'p_value':   round(p_value, 4),
                'Lags':      lags,
                'Crit_5pct': round(crit['5%'], 3),
                'Decision':  'STATIONARY (Keep Standard)' if p_value > 0.05 else 'TRENDING (Move to Decomp)',
                'Warning':   warning_msg,
            })
    return pd.DataFrame(results)


# 4. Execute the Audit
audit_results = stationarity_audit(df_pivot, standard_candidates)
audit_results

In [ ]:
import importlib
importlib.reload(af)

Pre-COVID   -> examine -> decide additive/multiplicative -> fit STL -> extract seasonal factors
COVID       -> apply pre-COVID factors -> measure deviation
Post-COVID  -> apply pre-COVID factors -> measure deviation
            -> separately examine if seasonal pattern has structurally changed

## Seasonality Test

In [ ]:
from statsmodels.tsa.seasonal import STL

def compute_seasonal_strength(data_df: pd.DataFrame, baseline: pd.DataFrame) -> pd.DataFrame:
    """
    Computes STL seasonal strength for each crime type and merges it into the baseline DataFrame.
    """

    def _strength(sub: pd.DataFrame) -> float:
        # Ensure sorted and numeric
        ts = (
            sub.sort_values('date')['crime_count']
            .astype(float)
            .to_numpy()
        )

        # STL requires at least 24 months (2 full seasonal cycles)
        if len(ts) < 24:
            return 0.0

        # Fit STL
        fit = STL(ts, period=12, robust=True).fit()

        R = fit.resid
        S = fit.seasonal

        # Compute variances safely
        var_R = np.nanvar(R)
        var_RS = np.nanvar(R + S)

        # Avoid division by zero
        if var_RS == 0 or np.isnan(var_RS):
            return 0.0

        strength = 1 - var_R / var_RS

        # Clamp to [0, 1]
        return float(max(0.0, min(1.0, strength)))

    # Compute seasonal strength per crime
    strengths = (
        data_df
        .groupby('fbi_code_desc')
        .apply(_strength, include_groups=False)
        .rename('seasonal_strength')
        .reset_index()
    )

    # Merge into baseline
    merged = baseline.merge(strengths, on='fbi_code_desc', how='left')

    return merged

In [ ]:
# Seasonal-Trend decomposition using Loess
baseline_df = compute_seasonal_strength(era_data_filled['pre_covid'], baseline_df)
# display
baseline_df

In [ ]:
def apply_seasonality_first_routing(baseline):

    baseline = baseline.copy()

    # Initialize all routing flags
    baseline['use_presence'] = False
    baseline['use_decomp']   = False
    baseline['use_robust']   = False
    baseline['use_standard'] = False

    for idx, row in baseline.iterrows():

        # 1. Presence-based crimes (sparse)
        if row['presence_rate'] < 30 or row['zero_mad'] or row['mean'] < 1:
            baseline.at[idx, 'use_presence'] = True
            continue

        # 2. Strong seasonality -> STL mandatory
        if row['seasonal_strength'] > 0.75:
            baseline.at[idx, 'use_decomp'] = True
            continue

        # 3. Moderate seasonality -> STL recommended
        if 0.40 < row['seasonal_strength'] <= 0.75:
            baseline.at[idx, 'use_decomp'] = True
            continue

        # 4. Weak/no seasonality -> CV decides
        if row['seasonal_strength'] <= 0.40:

            # High CV -> robust z-score
            if row['cv'] > 40:
                baseline.at[idx, 'use_robust'] = True
            else:
                baseline.at[idx, 'use_standard'] = True

    return baseline


In [ ]:
# Pull pre-COVID filled dataframe (contains the 0-filled rows for every crime/month)
total_pre_months = era_data_filled['pre_covid']['date'].nunique()  # Should be 230
# --- Execution ---
baseline_df = apply_seasonality_first_routing(era_data_filled['pre_covid'])
# af.print_baseline_report(baseline_df)

In [ ]:
# Extract available crimes for the seasonal amplitude plot (only those with decomp stats)
available_crimes = baseline_df.fbi_code_desc.to_list()
# Seasonal Amplitude Plot
plot.plot_seasonal_amplitude(era_data_filled['pre_covid'], available_crimes)

In [ ]:
era_data_filled['pre_covid'].columns


In [ ]:
# Global Mean-Variance Plot 
plot.plot_global_mean_variance(era_data_filled['pre_covid'], n_labels=5)

In [ ]:
#  Mean-Variance Analysis Plot
plot.plot_mean_variance_analysis(era_data_filled['pre_covid'], 'Drug Abuse Violations', window=12)

## Seasonal Decomposition (Pre‑COVID): What It Does and Why It Matters
> * Seasonal decomposition is essential for your caution‑tier crimes - the ones with CV between 30–60% because their variance is inflated by strong, repeating seasonal cycles. Without decomposition, the z‑gap would confuse seasonal peaks with structural breaks, producing misleading signals.
> * For crime types with moderate variance (CV 30–60%), raw Z‑scores are confounded by strong seasonal cycles that inflate variance and obscure structural breaks. To address this, we apply additive STL decomposition with a 12‑month period to each of the ten caution‑tier crimes. The decomposition separates each series into trend, seasonal, and residual components. The residuals, which represent the crime’s natural random noise after removing trend and seasonality, form the basis for the seasonally adjusted Z‑gap (sadj_z). By using the residual mean and residual standard deviation as the reference distribution, sadj_z isolates genuine deviations from baseline behavior and substantially increases sensitivity to COVID‑era and post‑COVID structural shifts.

* Seasonal decomposition is a time series technique that breaks your data into three core components so you can understand what’s driving patterns over time.
    * Observed = Trend + Seasonality + Residual
        * Trend: Long-term direction of the series
        * Seasonality: repeating 12-month cycle (e.g., summer peaks, winter troughs)
        * Residual (noise): the irregular fluctuation remaining after trend and seasonality are removed
        * 
* **Why is decomposition applied to 10 crimes**
    1. Seasonality inflates variance
        * The raw standard deviation is too large
        * Z‑gap denominator becomes inflated
        * real shifts look artificially small
    2. Seasonal peaks mimic structural breaks
        * Summer spikes look like "COVID effects"
        * Winter troughs look like "post‑COVID declines"
    
    Seasonal decomposition removes this confounding structure.
  
    3. How decomposition improves Z‑gap
        * After decomposition, you compute seasonally adjusted Z‑gap (sadj_z) using:
            * resid_mean instead of raw mean
            * resid_std instead of raw std

This is a huge improvement because residuals represent the crime’s natural random noise, not its full variance, where:
  * Seasonal peaks no longer inflate Z‑gap
  * Seasonal troughs no longer create false negatives
  * COVID‑driven deviations stand out sharply
  * Post‑COVID stabilization or reorganization becomes visible
    
`sadj_z` actually measures how far this month deviates from the crime’s natural noise floor, after removing trend and seasonality, which is a much more sensitive and interpretable metric than raw Z‑gap for seasonal crimes, where one can detect the following.
* abrupt COVID‑era shocks
* delayed post‑COVID rhythm changes
* regime shifts in cluster structure
* anomalies that would otherwise be masked by seasonal variance
  
> Applied to all 9 caution crimes plus Liquor Laws (10 total) using additive or multiplicative decomposition with a 12-month period. The decomposition separates each crime's baseline time series into three components:
> * The residuals become the new baseline for `sadj_z` (seasonally adjusted z-gap). By using `resid_mean` and `resid_std` instead of raw mean and std as the denominator, `sadj_z` measures deviation from the crime's natural random noise rather than from its full variance, making it dramatically more sensitive to genuine COVID-driven shifts.

## Mean vs Variance Relationship
| Pattern                    | Meaning                                     |
| -------------------------- | ------------------------------------------- |
| flat cloud                 | additive OK                                 |
| upward trend               | variance scales with level -> multiplicative |
| strong linear relationship | very strong multiplicative structure        |


In [ ]:
import numpy as np
from scipy.stats import linregress

log_mean = np.log(stats['mean'])
log_var  = np.log(stats['variance'])

slope, intercept, r, p, _ = linregress(log_mean, log_var)

print(f"slope = {slope:.2f}, R² = {r**2:.3f}")

## Correlation Check
Rule of thumb:
* corr > 0.6 -> strong scaling
* corr > 0.8 -> multiplicative likely

In [ ]:
corr, p = pearsonr(stats['mean'], stats['variance'])
print(f"corr = {corr:.3f}, p = {p:.3g}")

## Coefficient of Variation (CV) Stability

In [ ]:
stats['cv'] = stats['std'] / stats['mean']

## Log Transformation Test

In [ ]:
pre['log_crime'] = np.log1p(pre['crime_count'])

stats_log = (
    pre.groupby('fbi_code_desc')['log_crime']
    .agg(['mean', 'std'])
)

stats_log['variance'] = stats_log['std'] ** 2

## Time series visual (quick sanity check)

In [ ]:
crime = 'Gambling'  # example
subset = pre[pre['fbi_code_desc'] == crime]

plt.plot(subset['date'], subset['crime_count'])
plt.title(crime)
plt.show()

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Pivot pre-COVID to wide format (dates as rows, crimes as columns)
pre_wide = (
    pre.pivot(index='date', columns='fbi_code_desc', values='crime_count')
    .sort_index()
    .astype(float)
)

# Crimes that get decomposition (caution flag + Liquor Laws)
decomp_crimes = baseline_df[baseline_df['use_decomp'] == True]['fbi_code_desc'].tolist()
print(f"Decomposing {len(decomp_crimes)} crimes:")
for c in decomp_crimes:
    print(f"  - {c}")

# Run additive seasonal decomposition (period=12 months)
decomp_residuals = {}
for crime in decomp_crimes:
    series = pre_wide[crime]
    try:
        result = seasonal_decompose(
            series,
            model='additive',
            period=12,
            extrapolate_trend='freq'
        )
        resid = result.resid.dropna()
        decomp_residuals[crime] = {
            'resid_mean'  : resid.mean(),
            'resid_std'   : resid.std(),
            'resid_median': resid.median(),
            'resid_mad'   : (resid - resid.median()).abs().median()
        }
        print(f"  ✓ {crime}")
    except Exception as e:
        print(f"  ✗ {crime}: {e}")

# Build decomp dataframe - used as input to sadj_z in Stage 3
decomp_df = (
    pd.DataFrame(decomp_residuals)
    .T
    .reset_index()
    .rename(columns={'index': 'fbi_code_desc'})
    .round(4)
)

# Summary
print(f"\n=== Decomposition residuals ({len(decomp_df)} crimes) ===")
print(decomp_df.to_string(index=False))

print(f"\n=== Residual std vs original std ===")
comparison = decomp_df.merge(
    baseline_df[['fbi_code_desc', 'std']],
    on='fbi_code_desc'
)
comparison['std_reduction_pct'] = (
    (comparison['std'] - comparison['resid_std']) / comparison['std'] * 100
).round(1)
print(comparison[['fbi_code_desc', 'std', 'resid_std', 'std_reduction_pct']]
      .sort_values('std_reduction_pct', ascending=False)
      .to_string(index=False))

### Variance reduction is strong across the board

- Top 8 crimes had **62–84% of variance removed** - seasonal patterns were dominating their baselines, now stripped out
- Liquor Laws (71.7%) - substantial seasonal component confirmed; correct to include
- Homicide (48.8%) and Embezzlement (37.2%) - lower reduction, meaning their baseline variance is less seasonal and more genuinely random

## Compute Metrics Per Era

All metrics are computed for the COVID and post-COVID eras relative to the pre-COVID baseline.

$$z\_gap = \frac{\text{era\_mean} - \text{baseline\_mean}}{\text{baseline\_std}}$$

$$robust\_z = \frac{\text{era\_median} - \text{baseline\_median}}{1.4826 \times \text{baseline\_MAD}}$$

### Metric interpretation matrix

| Condition | Interpretation |
|---|---|
| High z_gap + high r_spike | Strong, statistically unusual shift in volume - most meaningful |
| High z_gap + low r_spike | Statistically unusual but small absolute change - common in stable low-volume crimes |
| Low z_gap + high r_spike | Large volume change but within normal variability - seen in noisy crimes |
| Low z_gap + low r_spike | No meaningful shift in either dimension |

This is exactly why using both together in the feature matrix gives the clustering more signal than either metric alone - z_gap captures statistical unusualness, r_spike captures magnitude, and together they describe the full shape of each crime's COVID response.

In [ ]:
# Compute metrics for COVID and post-COVID eras
cov_metrics  = af.compute_metrics(era_data_filled['covid'], 'covid', baseline_df, decomp_df)
post_metrics = af.compute_metrics(era_data_filled['post_covid'], 'post_covid', baseline_df, decomp_df)

# Combine into a single metrics table
metrics = pd.concat([cov_metrics, post_metrics], ignore_index=True)

# View columns
cols = [
    'fbi_code_desc', 'cv_flag',
    'z_gap', 'robust_z', 'sadj_z',
    'r_spike', 'pct_change', 'presence_rate'
]

print("=== COVID metrics (sorted by |z_gap|) ===")
print(
    cov_metrics[cols]
    .assign(abs_z=lambda x: x['z_gap'].abs())
    .sort_values('abs_z', ascending=False)
    .drop(columns='abs_z')
    .to_string(index=False)
)

print("\n=== Post-COVID metrics (sorted by |z_gap|) ===")
print(
    post_metrics[cols]
    .assign(abs_z=lambda x: x['z_gap'].abs())
    .sort_values('abs_z', ascending=False)
    .drop(columns='abs_z')
    .to_string(index=False)
)

## Four Patterns Emerge from the Metrics Table

**Pattern 1 - Crimes that spiked and stayed up**
Weapons Violations and Aggravated Assault both show positive z_gap in COVID and post-COVID. They went up during COVID and did not recover. Weapons Violations is the most extreme signal in the entire dataset (z_gap_covid = 4.81, +120%).

**Pattern 2 - Crimes that dropped and stayed down**
The majority of crimes fall here - Disorderly Conduct, Drug Abuse, Larceny, Burglary, Simple Battery, Robbery. All show negative z_gap in both eras, meaning the drop persisted into post-COVID. Drug Abuse (−86% COVID, −84% post-COVID) shows the strongest sustained suppression, likely driven by enforcement policy changes that never reversed.

**Pattern 3 - Crimes that spiked during COVID but recovered**
Homicide is the clearest example - sadj_z = 9.05 during COVID (+50%) but drops to sadj_z = 6.24 post-COVID (+3.5%). A classic COVID disruption/recovery arc. The seasonal decomposition was critical here - the raw z_gap of 1.54 dramatically underestimated the true spike magnitude.

**Pattern 4 - Near-elimination crimes**
Prostitution (−94% COVID, −94% post-COVID) and Gambling (−98% COVID, −98% post-COVID) essentially disappeared and never came back. These are likely enforcement and reporting driven rather than true behavioral change. Drug Abuse (−86%) also falls into this category.

### sadj_z vs z_gap divergence for caution crimes

Seasonal decomposition revealed that raw z_gap systematically underestimated the COVID impact for seasonally driven crimes:

| Crime | z_gap (COVID) | sadj_z (COVID) | Interpretation |
|---|---|---|---|
| Vandalism | −1.22 | 10.55 | Seasonal pattern masked a huge real drop |
| Motor Vehicle Theft | −0.44 | 11.12 | Same - decomp reveals much stronger signal |
| Aggravated Battery | −0.49 | 9.67 | Raw z_gap dramatically underestimated shift |
| Homicide | 1.54 | 9.05 | Seasonal decomp amplifies the true COVID spike |

> **Note on sadj_z magnitude:** Values like 10.55 appear large but are not directly comparable to z_gap values. sadj_z uses `resid_std` as its denominator - the residual standard deviation after removing seasonal variance - which is 62–84% smaller than the raw `baseline_std` used by z_gap. A sadj_z of 10.55 is not 10× more extreme than a z_gap of 1.22; it is measured on a finer scale. Both indicate a meaningful shift - sadj_z simply resolves it more precisely because seasonal noise has been stripped from the denominator.

## Visulaize

In [ ]:
# Weapons Violations - Pattern 1: spiked and stayed up
plot.plot_crime_history('Weapons Violations', baseline_df, era_data_filled)

In [ ]:
# Homicide - Pattern 3: spiked during COVID but recovered
plot.plot_crime_history('Homicide \u2013 1st or 2nd Degree', baseline_df, era_data_filled)

In [ ]:
# Larceny - Pattern 2: dropped and stayed down
plot.plot_crime_history('Larceny \u2013 Theft', baseline_df, era_data_filled)

In [ ]:
# Motor Vehicle Theft - caution crime: sadj_z reveals much stronger signal than z_gap
plot.plot_crime_history('Motor Vehicle Theft', baseline_df, era_data_filled)

In [ ]:
# Drug Abuse Violations - Pattern 4: near-elimination
plot.plot_crime_history('Drug Abuse Violations', baseline_df, era_data_filled)

## Build the Feature Matrix

A **26 × 12** feature matrix (26 crimes × 6 metrics × 2 eras), normalized and imputed, ready for both correlation and DTW clustering.

| Metric | Crimes it applies to |
|---|---|
| z_gap | All 26 |
| robust_z | 4 noisy crimes (NaN for others) |
| sadj_z | 10 caution + Liquor Laws (NaN for others) |
| r_spike | All 26 |
| pct_change | All 26 |
| presence_rate | All 26 |

> **Note on r_spike and pct_change:** These two metrics are mathematically identical after normalization - `pct_change = (r_spike − 1) × 100`. They are retained as separate columns because they serve different routing categories with non-overlapping weights: `r_spike` is the magnitude signal for reliable, use_decomp, and use_robust crimes (weight = 0.75); `pct_change` is the magnitude signal for use_presence crimes only (weight = 1.0, where r_spike = 0.0). Together, they function as a single-magnitude signal split across two routing paths, not as redundant information.
> * A `routing category` is a label that determines how a feature is processed, weighted, and interpreted within different analytical paths of a modeling pipeline.

## Assign a Weight to Each Feature

We already have a principled basis for weighting built into the pipeline - the cv_flag and routing flags - and we know which metrics are more reliable for which crimes. Weighting formalizes that knowledge into the distance calculation so the clustering algorithm respects the same statistical logic applied in Stages 2 and 3.

### Principled approach - per-crime weights

**For reliable crimes** (z_gap primary):
- `z_gap` -> 2.0 (primary - std is a reliable unit of measurement)
- `r_spike` -> 0.75 (secondary - magnitude context)
- `presence_rate` -> 0.25 (supplementary floor)
- Everything else -> 0.0 (imputed or redundant)

**For use_decomp crimes** (sadj_z primary):
- `sadj_z` -> 2.0 (primary - seasonal variance stripped, most sensitive signal)
- `z_gap` -> 1.0 (secondary - still carries directional information)
- `r_spike` -> 0.75 (secondary - magnitude context)
- `presence_rate` -> 0.25 (supplementary floor)
- Everything else -> 0.0

**For use_robust crimes** (robust_z primary):
- `robust_z` -> 2.0 (primary - resistant to outliers, honest for noisy crimes)
- `r_spike` -> 0.75 (secondary - magnitude context)
- `presence_rate` -> 0.25 (supplementary floor)
- Everything else -> 0.0

**For use_presence crimes** (Involuntary Manslaughter only):
- `presence_rate` -> 2.0 (primary - % of months with at least one incident)
- `pct_change` -> 1.0 (secondary - magnitude when it did occur)
- `z_gap` -> 0.25 (floor - unreliable but not zero)
- `robust_z` -> 0.0 (MAD = 0, undefined)
- `sadj_z` -> 0.0 (not applicable, no decomposition)
- `r_spike` -> 0.0 (pct_change already captures this)

### Final weight table

| Feature | reliable | use_decomp | use_robust | use_presence |
|---|---|---|---|---|
| z_gap | 2.0 | 1.0 | 0.0 | 0.25 |
| robust_z | 0.0 | 0.0 | 2.0 | 0.0 |
| sadj_z | 0.0 | 2.0 | 0.0 | 0.0 |
| r_spike | 0.75 | 0.75 | 0.75 | 0.0 |
| pct_change | 0.0 | 0.0 | 0.0 | 1.0 |
| presence_rate | 0.25 | 0.25 | 0.25 | 2.0 |

### Why 0.0 for unmentioned features

Every 0.0 corresponds to a metric that was either imputed (NaN filled with column median - not real data for that crime), unreliable for that crime's statistical profile, or redundant with another metric already included. Setting them to 0.0 ensures that clustering uses only metrics genuinely computed for each crime.

### Winsorizing before weighting

Extreme values are capped at $\pm$3.0 before applying weights to prevent amplification of outlier weights. Three crimes had values capped:

- **Involuntary Manslaughter** - 4 features (presence_rate, r_spike, pct_change)
- **Weapons Violations** - 4 features (z_gap, robust_z both eras)
- **Motor Vehicle Theft** - 1 feature (sadj_z_post_covid)

> **Note:** The domination check uses Euclidean distance for diagnostic purposes only. The actual clustering uses correlation and DTW distances, which are scale-invariant and pattern-based, respectively. Euclidean distance is not used in the clustering pipeline.

In [ ]:
# ============================================================
# Build the Feature Matrix
# ============================================================
from scipy.spatial.distance import cdist

# Step 1: pivot metrics into wide format
# Each crime gets one row with all metrics from both eras as columns
feature_matrix = metrics.pivot(
    index='fbi_code_desc',
    columns='era',
    values=['z_gap', 'robust_z', 'sadj_z', 'r_spike', 'pct_change', 'presence_rate']
)

# Flatten multi-level column names into single strings
# e.g. ('z_gap', 'covid') -> 'z_gap_covid'
feature_matrix.columns = [f"{metric}_{era}" for metric, era in feature_matrix.columns]
feature_matrix         = feature_matrix.reset_index()

print("=" * 70)
print("STEP 1: Raw feature matrix")
print("=" * 70)
print(f"Shape : {feature_matrix.shape[0]} crimes × {feature_matrix.shape[1]-1} features")
print(f"NaNs  :")
print(feature_matrix.isnull().sum()[feature_matrix.isnull().sum() > 0].to_string())

# Step 2: separate crime labels from features
crime_labels = feature_matrix['fbi_code_desc'].values
feature_cols = [c for c in feature_matrix.columns if c != 'fbi_code_desc']
X_raw        = feature_matrix[feature_cols].copy()

# Step 3: impute NaNs
# sadj_z NaNs (16 per era): metric is not applicable for non-decomp crimes
#    impute with 0 (no seasonal deviation = neutral, not "typical decomp crime")
# robust_z NaNs (1 per era): Involuntary Manslaughter has zero MAD -> division undefined
#    impute with column median (crime is atypical; median is the best neutral proxy)
#
# Note on pct_change vs r_spike:
#   Both are linear transformations of (era_mean / baseline_mean) and are
#   mathematically identical after StandardScaler normalization.
#   They are NOT redundant in the feature matrix because the weight profiles
#   assign them to non-overlapping routing categories:
#     r_spike    -> active (weight=0.75) for reliable / use_decomp / use_robust
#     pct_change -> active (weight=1.00) for use_presence only
#   Together they act as a single magnitude signal routed by crime type. 
X_imputed = X_raw.copy()

# Columns that represent sadj_z (inapplicable for non-decomp crimes -> 0)
sadj_cols = [c for c in feature_cols if c.startswith('sadj_z')]

# Columns that represent robust_z (undefined for zero-MAD crime -> median)
robust_z_cols = [c for c in feature_cols if c.startswith('robust_z')]

# All remaining columns (no NaNs expected, but median-fill as safety net)
other_cols = [c for c in feature_cols if c not in sadj_cols + robust_z_cols]

# sadj_z -> 0 (not applicable, not missing)
for col in sadj_cols:
    n_nan = X_imputed[col].isna().sum()
    X_imputed[col] = X_imputed[col].fillna(0)
    if n_nan > 0:
        print(f"  sadj_z imputed with 0 : {col}  ({n_nan} NaNs)")

# robust_z -> column median (undefined due to zero MAD, not inapplicable)
for col in robust_z_cols:
    median_val = X_imputed[col].median()
    n_nan = X_imputed[col].isna().sum()
    X_imputed[col] = X_imputed[col].fillna(median_val)
    if n_nan > 0:
        print(f"  robust_z imputed with median ({median_val:.4f}) : {col}  ({n_nan} NaNs)")

# Other columns -> median as safety net (should be 0 NaNs)
for col in other_cols:
    median_val = X_imputed[col].median()
    X_imputed[col] = X_imputed[col].fillna(median_val)

print("\n" + "=" * 70)
print("STEP 2: After median imputation")
print("=" * 70)
print(f"\nNaNs remaining after imputation: {X_imputed.isnull().sum().sum()}")


# Step 4: z-score normalize (so pct_change doesn't dominate)
# Puts all metrics on the same scale so pct_change doesn't dominate
scaler   = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X_imputed),
    columns=feature_cols,
    index=crime_labels
)

print("\n" + "=" * 70)
print("STEP 3: After z-score normalization")
print("=" * 70)
print(f"Mean  ≈ 0: {(X_scaled.mean().abs() < 0.01).all()}")
print(f"Std   ≈ 1: {(X_scaled.std().round(2) == 1.02).all()}")

# Step 5: winsorize at ±3.0
# Caps extreme values before weighting to prevent weight amplification of outliers
WINSOR_CAP = 3.0
X_winsor   = X_scaled.clip(lower=-WINSOR_CAP, upper=WINSOR_CAP)

print("\n" + "=" * 70)
print("STEP 4: After winsorizing (cap = ±3.0)")
print("=" * 70)
extreme_before = (X_scaled.abs() > WINSOR_CAP).sum().sum()
extreme_after  = (X_winsor.abs() > WINSOR_CAP).sum().sum()
print(f"Values beyond ±3.0 before : {extreme_before}")
print(f"Values beyond ±3.0 after  : {extreme_after}")
print(f"\nCapped values per crime:")
capped = (X_scaled.abs() > WINSOR_CAP)
for crime in crime_labels:
    n = capped.loc[crime].sum()
    if n > 0:
        features = capped.columns[capped.loc[crime]].tolist()
        print(f"  {crime:<50} {n} feature(s): {features}")

# Step 6: define weight profiles per routing category
weight_profiles = {
    'reliable': {
        'z_gap_covid'              : 2.00,
        'z_gap_post_covid'         : 2.00,
        'robust_z_covid'           : 0.00,
        'robust_z_post_covid'      : 0.00,
        'sadj_z_covid'             : 0.00,
        'sadj_z_post_covid'        : 0.00,
        'r_spike_covid'            : 0.75,
        'r_spike_post_covid'       : 0.75,
        'pct_change_covid'         : 0.00,
        'pct_change_post_covid'    : 0.00,
        'presence_rate_covid'      : 0.25,
        'presence_rate_post_covid' : 0.25,
    },
    'use_decomp': {
        'z_gap_covid'              : 1.00,
        'z_gap_post_covid'         : 1.00,
        'robust_z_covid'           : 0.00,
        'robust_z_post_covid'      : 0.00,
        'sadj_z_covid'             : 2.00,
        'sadj_z_post_covid'        : 2.00,
        'r_spike_covid'            : 0.75,
        'r_spike_post_covid'       : 0.75,
        'pct_change_covid'         : 0.00,
        'pct_change_post_covid'    : 0.00,
        'presence_rate_covid'      : 0.25,
        'presence_rate_post_covid' : 0.25,
    },
    'use_robust': {
        'z_gap_covid'              : 0.00,
        'z_gap_post_covid'         : 0.00,
        'robust_z_covid'           : 2.00,
        'robust_z_post_covid'      : 2.00,
        'sadj_z_covid'             : 0.00,
        'sadj_z_post_covid'        : 0.00,
        'r_spike_covid'            : 0.75,
        'r_spike_post_covid'       : 0.75,
        'pct_change_covid'         : 0.00,
        'pct_change_post_covid'    : 0.00,
        'presence_rate_covid'      : 0.25,
        'presence_rate_post_covid' : 0.25,
    },
    'use_presence': {
        'z_gap_covid'              : 0.25,
        'z_gap_post_covid'         : 0.25,
        'robust_z_covid'           : 0.00,
        'robust_z_post_covid'      : 0.00,
        'sadj_z_covid'             : 0.00,
        'sadj_z_post_covid'        : 0.00,
        'r_spike_covid'            : 0.00,
        'r_spike_post_covid'       : 0.00,
        'pct_change_covid'         : 1.00,
        'pct_change_post_covid'    : 1.00,
        'presence_rate_covid'      : 2.00,
        'presence_rate_post_covid' : 2.00,
    },
}

# Step 7: assign routing category per crime
# Priority: use_presence -> use_decomp -> use_robust -> reliable
def get_routing(row):
    if row['use_presence']:
        return 'use_presence'
    elif row['use_decomp']:
        return 'use_decomp'
    elif row['use_robust']:
        return 'use_robust'
    else:
        return 'reliable'

baseline_df['routing'] = baseline_df.apply(get_routing, axis=1)

print("\n" + "=" * 70)
print("STEP 5: Routing assignment")
print("=" * 70)
print(
    baseline_df[['fbi_code_desc', 'cv_flag', 'routing']]
    .sort_values('routing')
    .to_string(index=False)
)
print(f"\nRouting breakdown: {baseline_df['routing'].value_counts().to_dict()}")

# Step 8: build per-crime weight matrix
weight_matrix = pd.DataFrame(
    index=crime_labels,
    columns=feature_cols,
    dtype=float
)
for crime in crime_labels:
    routing = baseline_df.loc[
        baseline_df['fbi_code_desc'] == crime, 'routing'
    ].values[0]
    for feature in feature_cols:
        weight_matrix.loc[crime, feature] = weight_profiles[routing][feature]

# Step 9: apply weights to winsorized matrix
# Winsorize first, then weight - prevents weight amplification of extremes
X_weighted = X_winsor * weight_matrix

# Step 10: display with short column names
display_cols = {
    'z_gap_covid'              : 'zgap_c',
    'z_gap_post_covid'         : 'zgap_p',
    'robust_z_covid'           : 'robz_c',
    'robust_z_post_covid'      : 'robz_p',
    'sadj_z_covid'             : 'sadj_c',
    'sadj_z_post_covid'        : 'sadj_p',
    'r_spike_covid'            : 'rspk_c',
    'r_spike_post_covid'       : 'rspk_p',
    'pct_change_covid'         : 'pct_c',
    'pct_change_post_covid'    : 'pct_p',
    'presence_rate_covid'      : 'pres_c',
    'presence_rate_post_covid' : 'pres_p',
}

print("\n" + "=" * 70)
print("STEP 6: Weight matrix")
print("=" * 70)
print(weight_matrix.rename(columns=display_cols).to_string())

print("\n" + "=" * 70)
print("STEP 7: Weighted feature matrix (winsorized + weighted)")
print("=" * 70)
print(X_weighted.rename(columns=display_cols).round(2).to_string())

# Step 11: domination check
# Note: Euclidean distances used here for diagnostic purposes only.
# Actual clustering uses correlation and DTW distances.
print("\n" + "=" * 70)
print("STEP 8: Domination check - Euclidean distance from outlier crimes")
print("(diagnostic only - clustering uses correlation and DTW distances)")
print("=" * 70)

print("Extreme values controlled (±3.0 cap):")
print(f"  Before winsorizing: {(X_scaled.abs() > 3.0).sum().sum()} values beyond ±3.0")
print(f"  After  winsorizing: {(X_winsor.abs() > 3.0).sum().sum()} values beyond ±3.0")

for crime, label in [
    ('Involuntary Manslaughter / Reckless Homicide', 'Involuntary Manslaughter'),
    ('Weapons Violations',                           'Weapons Violations'),
    ('Gambling',                                     'Gambling'),
]:
    dist = pd.Series(
        cdist(X_weighted.loc[[crime]], X_weighted)[0],
        index=X_weighted.index
    ).sort_values()
    print(f"\nEuclidean distance from {label} (top 5):")
    print(dist.head(6).round(2).to_string())

print("\n" + "=" * 70)
print("X_weighted ready for correlation and DTW distance computation")
print("=" * 70)
print(f"Shape : {X_weighted.shape[0]} crimes × {X_weighted.shape[1]} features")
print(f"NaNs  : {X_weighted.isnull().sum().sum()}")

## Correlation and DTW Distance

### What each distance metric measures

**Correlation distance** = 1 − Pearson correlation
- Range: [0, 2] - 0 = identical direction, 1 = uncorrelated, 2 = opposite direction
- Measures how similarly two crimes' weighted feature vectors are shaped
- Scale-invariant - direction and relative magnitude matter, absolute count levels do not
- Two crimes with identical z_gap and r_spike profiles across both eras will have correlation distance ≈ 0 regardless of monthly volume
- **Answers:** Which crimes responded to COVID in statistically similar ways?

**DTW distance** (Dynamic Time Warping)
- Measures similarity of monthly count time series with elastic alignment
- Sakoe-Chiba band = 10% of series length - constrains warping to prevent pathological alignments
- Length normalized per era for fair comparison across pre (230 months), COVID (34), post (36)
- **Answers:** Which crimes had similar monthly count trajectories?

### DTW Option B - era-separated (primary)

$$total\_dtw = 0.15 \times \frac{dtw(pre_A, pre_B)}{230} + 0.55 \times \frac{dtw(covid_A, covid_B)}{34} + 0.30 \times \frac{dtw(post_A, post_B)}{36}$$

Explicitly upweights the COVID era (55%) - consistent with the era-structured analysis pipeline.

### DTW Option A - concatenated (cross-check)

All 300 months as one series. Pre-COVID dominates (77% of the series). Used only to validate that DTW-B findings are not an artifact of era weighting.

### Linkage methods tested

Three linkage methods applied to each distance matrix:
- **Single** - distance between closest pair - tends to chain
- **Complete** - distance between furthest pair - compact spherical clusters
- **Average** - mean distance between all pairs - balanced, recommended default

Ward linkage is not applicable here - requires Euclidean distance only.

### Cluster selection criteria

Two independent methods per linkage/metric combination:
- **Inconsistency coefficient** - flags where merge distances become statistically unusual relative to local merge history (threshold search: 0.7 -> 1.25)
- **Linkage gap** - finds the largest jump in the merge distance sequence (natural elbow)
- **Consensus k** - final k where both methods agree; flagged as ambiguous if they disagree

In [ ]:
# ============================================================
# DTW Distance Computation
# ============================================================

# Helper: distance matrix stats
def distance_stats(dist_df, label):
    flat = dist_df.values[np.triu_indices(len(dist_df), k=1)]
    print(f"\n{label}:")
    print(f"  Shape          : {dist_df.shape}")
    print(f"  Min            : {flat.min():.4f}")
    print(f"  Max            : {flat.max():.4f}")
    print(f"  Mean           : {flat.mean():.4f}")
    print(f"  Median         : {np.median(flat):.4f}")
    print(f"  Std            : {flat.std():.4f}")
    print(f"  Diagonal zeros : {(np.diag(dist_df.values) == 0).all()}")
    print(f"  Symmetric      : {np.allclose(dist_df.values, dist_df.values.T)}")
    print(f"  NaNs           : {dist_df.isnull().sum().sum()}")

# Helper: distance matrix heatmap
def plot_distance_heatmap(dist_df, title):
    labels    = list(dist_df.columns)
    n         = len(labels)
    cell_size = 0.4
    fig, ax   = plt.subplots(figsize=(n * cell_size, n * cell_size))
    im        = ax.imshow(dist_df.values, cmap='coolwarm_r', aspect='auto')
    cbar      = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Distance', fontsize=10)
    ax.set_xticks(np.arange(n))
    ax.set_yticks(np.arange(n))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_xticks(np.arange(n) - 0.5, minor=True)
    ax.set_yticks(np.arange(n) - 0.5, minor=True)
    ax.grid(which='minor', color='white', linewidth=0.5)
    ax.set_aspect('equal', adjustable='box')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=15)
    plt.tight_layout()
    plt.show()

# Helper: closest neighbors
def print_closest_neighbors(dist_df, crime, label, top_n=3):
    row     = dist_df.loc[crime].drop(crime)
    nearest = row.nsmallest(top_n)
    first   = True
    for neighbor, dist in nearest.items():
        met_lbl = label if first else ''
        print(f"  {met_lbl:<20} {neighbor:<50} {dist:.4f}")
        first = False

# Part 1: Correlation distance matrix
# Input  : X_weighted (26 × 12 weighted feature matrix)
# Metric : 1 - Pearson correlation
# Range  : [0, 2] - 0=identical, 1=uncorrelated, 2=opposite
print("=" * 70)
print("PART 1: Correlation distance matrix")
print("=" * 70)

corr_dist = hc.compute_correlation_distance_matrix(
    X_weighted,
    labels=list(crime_labels)
)

distance_stats(corr_dist, "Correlation distance")
plot_distance_heatmap(corr_dist, "Correlation distance matrix\n(Blue = similar, Red = dissimilar)")

# Part 2: DTW distance matrix - Option B (primary)
# Era-separated with weights: pre=0.15, covid=0.55, post=0.30
# Normalized by series length within crime_era_clustering.py
# Sakoe-Chiba band = 10% of series length per era
print("\n" + "=" * 70)
print("PART 2: DTW distance matrix - Option B (era-separated, primary)")
print("=" * 70)
print(f"Era weights : {hc.DEFAULT_ERA_WEIGHTS}")
print(f"Warping pct : {hc.DEFAULT_WARPING_PCT}")

dtw_dist_B = hc.compute_dtw_distance_matrix(
    era_data_filled = era_data_filled,
    crime_labels    = list(crime_labels),
    era_weights     = hc.DEFAULT_ERA_WEIGHTS,
    warping_pct     = hc.DEFAULT_WARPING_PCT,
    option          = 'B'
)

distance_stats(dtw_dist_B, "DTW-B distance (era-separated)")
plot_distance_heatmap(dtw_dist_B, "DTW distance matrix - Option B: era-separated (primary)\n(Blue = similar, Red = dissimilar)")

# Part 3: DTW distance matrix - Option A (cross-check)
# All 300 months concatenated as one series per crime
print("\n" + "=" * 70)
print("PART 3: DTW distance matrix - Option A (concatenated, cross-check)")
print("=" * 70)
print(f"Warping pct : {hc.DEFAULT_WARPING_PCT}")

dtw_dist_A = hc.compute_dtw_distance_matrix(
    era_data_filled = era_data_filled,
    crime_labels    = list(crime_labels),
    era_weights     = hc.DEFAULT_ERA_WEIGHTS,
    warping_pct     = hc.DEFAULT_WARPING_PCT,
    option          = 'A'
)

distance_stats(dtw_dist_A, "DTW-A distance (concatenated)")
plot_distance_heatmap(dtw_dist_A, "DTW distance matrix - Option A: concatenated (cross-check)\n(Blue = similar, Red = dissimilar)")

# Part 4: Distance matrix comparison - Spearman rank correlation
print("\n" + "=" * 70)
print("PART 4: Distance matrix comparison - Spearman rank correlation")
print("=" * 70)

idx        = np.triu_indices(len(crime_labels), k=1)
c_flat     = corr_dist.values[idx]
dtw_B_flat = dtw_dist_B.values[idx]
dtw_A_flat = dtw_dist_A.values[idx]

rho_corr_dtwB, p_corr_dtwB = spearmanr(c_flat,     dtw_B_flat)
rho_corr_dtwA, p_corr_dtwA = spearmanr(c_flat,     dtw_A_flat)
rho_dtwB_dtwA, p_dtwB_dtwA = spearmanr(dtw_B_flat, dtw_A_flat)

print(f"\n{'Comparison':<35} {'rho':>8}  {'p-value':>10}  {'Sig':>4}  Interpretation")
print("-" * 85)

for label, rho, p in [
    ('Correlation vs DTW-B', rho_corr_dtwB, p_corr_dtwB),
    ('Correlation vs DTW-A', rho_corr_dtwA, p_corr_dtwA),
    ('DTW-B vs DTW-A',       rho_dtwB_dtwA, p_dtwB_dtwA),
]:
    if rho >= 0.8:   interp = 'Strong agreement'
    elif rho >= 0.6: interp = 'Moderate agreement'
    elif rho >= 0.4: interp = 'Weak agreement'
    else:            interp = 'Low agreement - captures different structure'
    sig = 'Y' if p < 0.05 else 'N'
    print(f"{label:<35} {rho:>8.3f}  {p:>10.4f}  {sig:>4}  {interp}")

# Part 5: Closest neighbors - outlier crimes
print("\n" + "=" * 70)
print("PART 5: Closest neighbors - outlier crimes (top 3 per metric)")
print("=" * 70)

outlier_crimes = [
    'Involuntary Manslaughter / Reckless Homicide',
    'Weapons Violations',
    'Gambling',
]

print(f"\n  {'Metric':<20} {'Neighbor':<50} {'Distance':>8}")
print(f"  {'-'*80}")

for crime in outlier_crimes:
    print(f"\n  {crime}:")
    for label, dist_df in [
        ('Correlation', corr_dist),
        ('DTW-B',       dtw_dist_B),
        ('DTW-A',       dtw_dist_A),
    ]:
        print_closest_neighbors(dist_df, crime, label, top_n=3)

print("\n" + "=" * 70)
print("Distance matrices ready for Clustering")
print("=" * 70)
print(f"  corr_dist  : {corr_dist.shape}  correlation distance (feature vectors)")
print(f"  dtw_dist_B : {dtw_dist_B.shape}  DTW era-separated, primary")
print(f"  dtw_dist_A : {dtw_dist_A.shape}  DTW concatenated, cross-check")
print(f"\n  Key finding:")
print(f"  DTW-B vs DTW-A     rho = {rho_dtwB_dtwA:.3f}   - strong agreement")
print(f"  Correlation vs DTW rho ~ 0       - captures different structure")
print(f"\n  Validation:")
print(f"  Diagonal zeros : True")
print(f"  Symmetric      : True")
no_nans = not any([corr_dist.isnull().any().any(), dtw_dist_B.isnull().any().any(), dtw_dist_A.isnull().any().any()])
print(f"  No NaNs        : {no_nans}")

## Summary - Distance Computation

### Three distance matrices computed

| Matrix | Metric | Input | Purpose |
|---|---|---|---|
| `corr_dist` | Correlation | X_weighted (26×12) | Statistical response similarity |
| `dtw_dist_B` | DTW era-separated | Monthly counts, era-weighted | Count pattern similarity (primary) |
| `dtw_dist_A` | DTW concatenated | Monthly counts, 300 months | Count pattern similarity (cross-check) |

### Key findings

**Finding 1 - DTW-A and DTW-B strongly agree (ρ = 0.882, p < 0.0001)**  
The era-separated weighted approach and the concatenated approach produce nearly identical crime similarity rankings. DTW findings are robust - not an artifact of era structure or weights. The underlying count pattern similarity is stable.

**Finding 2 - Correlation and DTW fundamentally disagree (ρ ≈ 0, not significant)**  
Statistical response similarity and raw count pattern similarity are completely independent dimensions. This is valuable - not a problem:
- Crimes clustering together under **both** metrics = most robustly similar
- Crimes clustering differently = dimension-specific findings worth investigating

**Finding 3 - Count magnitude drives DTW clustering**  
Low-count crimes (Embezzlement, Stolen Property, Gambling, Involuntary Manslaughter) cluster tightly under DTW regardless of statistical profiles. High-volume crimes (Larceny ~6,500/month) are isolated. An inherent property of DTW on raw counts.

**Finding 4 - Statistical profile drives correlation clustering**  
Weapons Violations and Fraud are nearly identical under correlation (distance = 0.006) - both increased during COVID and stayed elevated. Gambling and Prostitution are nearly identical (distance = 0.12) - both near-eliminated across both eras.

### Per-era DTW hypothesis test

| Comparison | ρ | Rhythm shift | Sig |
|---|---|---|---|
| Pre vs COVID | 0.8086 | 0.1914 | ✓ |
| Pre vs Post-COVID | 0.8220 | 0.1780 | ✓ |
| COVID vs Post-COVID | 0.9749 | - | ✓ |

**Result:** Original hypothesis **NOT supported** at aggregate level.  
Temporal rhythm reorganization occurred **immediately during COVID** and persisted unchanged into post-COVID (COVID vs post-COVID ρ = 0.975).

### Refined research question

> COVID-19 simultaneously reorganized both the directional co-movement and the temporal rhythms of crime types. This reorganization persisted into the post-COVID period without further aggregate structural change; however, individual crime types may exhibit heterogeneous rhythm trajectories, with some showing delayed reorganization concentrated in the post-COVID period.

In [ ]:
baseline_df

In [ ]:
def apply_weight_routing(baseline_df):
    """
    Assigns analytical weights to crimes while ensuring mutual exclusivity 
    between r_spike and pct_change to prevent double-counting.
    """
    # Initialize weights
    baseline_df['w_rspike'] = 0.0
    baseline_df['w_pctchng'] = 0.0

    # 1. Route to r_spike: Reliable, Caution (Decomp), or Noisy (Robust)
    # Note: 'reliable' is a value in 'cv_flag', not a column name.
    mask_rspike = (
        (baseline_df['cv_flag'] == 'reliable') | 
        (baseline_df['use_decomp'] == True) | 
        (baseline_df['use_robust'] == True)
    )
    baseline_df.loc[mask_rspike, 'w_rspike'] = 0.75

    # 2. Route to pct_change: Rare/Presence crimes only
    # This ensures a crime cannot be both.
    mask_presence = (baseline_df['use_presence'] == True) & (~mask_rspike)
    baseline_df.loc[mask_presence, 'w_pctchng'] = 1.0

    # 3. Final Check for Leaky Abstractions
    double_dipped = baseline_df[(baseline_df['w_rspike'] > 0) & (baseline_df['w_pctchng'] > 0)]
    if not double_dipped.empty:
        raise ValueError(f"Logic Leak! The following crimes have dual weights: {double_dipped['fbi_code_desc'].tolist()}")
    
    print(f"Weight Routing Complete. Mutual Exclusivity Verified.")
    return baseline_df

# baseline_df = apply_weight_routing(baseline_df)

In [ ]:
baseline_df = apply_weight_routing(baseline_df)

In [ ]:
baseline_df

In [ ]:
# You should add this verification cell:
active_both = []
for crime in crime_labels:
    row = baseline_df[baseline_df['fbi_code_desc'] == crime].iloc[0]
    w_rspike   = 0.75 if (row['reliable'] or row['use_decomp'] or row['use_robust']) else 0
    w_pctchng  = 1.0  if row['use_presence'] else 0
    if w_rspike > 0 and w_pctchng > 0:
        active_both.append(crime)
print(f"Crimes where BOTH r_spike and pct_change are active: {active_both}")
# This list should be EMPTY. If it's not, you have double-counting.

## Save to File

In [ ]:
# # Save all distance matrices to a single feather file
# # Format: long/tidy - one row per crime pair per matrix
# # Columns: matrix, crime_a, crime_b, distance
# # Full matrix saved (both triangles), so reconstruction is a clean pivot()
# # Path mirrors existing data file: ../Data/crime_data_covid.feather

# file_list = [corr_dist, dtw_dist_B, dtw_dist_A]
# names     = ['corr_dist', 'dtw_dist_B', 'dtw_dist_A']

# long_frames = []
# for dist_df, name in zip(file_list, names):
#     # reset_index() brings crime_labels in as a column called 'fbi_code_desc'
#     melted = (
#         dist_df
#         .reset_index()
#         .rename(columns={'index': 'crime_a'})     # row crime
#         .melt(
#             id_vars   = 'crime_a',
#             var_name  = 'crime_b',                # col crime
#             value_name= 'distance'
#         )
#     )
#     melted['matrix'] = name
#     long_frames.append(melted)

# dist_long = (
#     pd.concat(long_frames, ignore_index=True)
#     [['matrix', 'crime_a', 'crime_b', 'distance']]  # enforce column order
# )

# PATH = '../Data/crime_distance_matrices.feather'
# feather.write_feather(dist_long, PATH)

# # Validation──
# loaded   = feather.read_feather(PATH)
# expected = 3 * 26 * 26   # 3 matrices × 26×26 full grid = 2028 rows

# assert len(loaded) == expected,      f"Row count wrong: {len(loaded)} ≠ {expected}"
# assert loaded['distance'].isna().sum() == 0, "NaNs found in saved distances"
# assert set(loaded['matrix'].unique()) == set(names), "Matrix names mismatch"

# print(f"Saved   : {PATH}")
# print(f"Shape   : {loaded.shape}  ({len(names)} matrices × 26 × 26)")
# print(f"Columns : {list(loaded.columns)}")
# print(f"Matrices: {sorted(loaded['matrix'].unique())}")
# print(f"NaNs    : {loaded['distance'].isna().sum()}")
# print(f"Diagonal zeros: {(loaded[loaded['crime_a'] == loaded['crime_b']]['distance'] == 0).all()}")